# Empathetic Conversational Recommender Systems — Experiment Notebook

Notebook **simple et modulaire** qui re-execute l'experience de l'article :

> Xiaoyu Zhang, Ruobing Xie, Yougang Lyu, Xin Xin, Pengjie Ren, Mingfei Liang, Bo Zhang, Zhanhui Kang, Maarten de Rijke, Zhaochun Ren.  
> **Towards Empathetic Conversational Recommender Systems**. *RecSys '24*, Bari, Italy — [doi:10.1145/3640457.3688133](https://doi.org/10.1145/3640457.3688133).

Le code de reference est celui du depot officiel [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR). Ce notebook se structure comme l'exemple [WebSemantic-Projet-n-1/projet_session/tag_reco_experiment.ipynb](https://github.com/WebSemantic-Projet-n-1/projet_session/blob/main/tag_reco_experiment.ipynb) :

- **Une fonction par cellule** (pas de logique inline).
- **Une seule cellule d'import de donnees**.
- **Tous les graphiques** sont factorises dans `src/viz/plots.py`.
- **Derniere cellule** : compilation des resultats et visualisations comparatives.

### Plan du notebook

1. Parametrages globaux (hyperparametres de l'article, flags d'execution).
2. Preparation de l'environnement (clone du depot ECR + archives `emo_data` / `ckpt`).
3. Pretraitement du dataset **ReDial** (Section 4.1).
4. Sous-tache **Emotional Semantic Fusion** — `train_pre.py` (Section 4.1).
5. Sous-tache **Emotion-aware Item Recommendation** — `train_rec.py` (Section 4.2).
6. Sous-tache **Emotion-aligned Response Generation** — `train_emp.py` + `infer_emp.py` (Section 4.3).
7. Chargement des metriques objectives / subjectives (Tables 1-3).
8. Pipeline principal (import des donnees en **une seule cellule**).
9. Compilation finale + visualisations comparatives.

> **Note pratique.** L'entrainement complet de ECR demande un GPU 24 GB et plusieurs heures. Un drapeau `DRY_RUN` permet de ne lancer que la preparation legere du pipeline tout en reproduisant fidelement les tableaux de l'article (valeurs de repli).

In [1]:
# Cellule 1 - Detection materielle (aligne avec la stack PyTorch de l'article)
import os

FORCE_CPU = False

if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

try:
    import torch
    _torch_ok = True
except ImportError:
    _torch_ok = False

if _torch_ok:
    has_cuda = (not FORCE_CPU) and torch.cuda.is_available()
    device = "cuda" if has_cuda else "cpu"
    print(f"torch {torch.__version__} - device: {device}")
    if has_cuda:
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        print(f"Memoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    a = torch.randn(1024, 1024, device=device)
    _ = (a @ a).sum().item()
    print(f"Sanity matmul OK sur {device}.")
else:
    print("torch n'est pas installe - installez-le avec `pip install -r requirements.txt` si vous souhaitez re-entrainer les modeles.")

torch 2.10.0+cu128 - device: cuda
CUDA device: NVIDIA GeForce RTX 5090
Memoire GPU: 33.7 GB
Sanity matmul OK sur cuda.


In [2]:
# Cellule 2 - Imports (stack notebook + visualisations)
import sys
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.viz.plots import (
    # Plots "article" (Tables 1-3, EDA Section 5.1)
    plot_ablation_study,
    plot_emotion_label_distribution,
    plot_feedback_distribution,
    plot_feedback_weights,
    plot_hyperparam_sweep,
    plot_llm_vs_human_correlation,
    plot_model_rankings,
    plot_objective_metrics,
    plot_review_coverage,
    plot_subjective_metrics,
    plot_subjective_radar,
    plot_training_loss,
    # Plots exploitant results/run_*.csv (accumulation multi-runs)
    plot_config_hash_trajectory,
    plot_dataset_diagnostic,
    plot_generation_quality,
    plot_phase_timings,
    plot_pre_convergence,
    plot_runs_vs_article,
    # Dispatch persistance
    set_save_dir,
)

print(f"Project root: {ROOT}")

Project root: /workspace


## 1. Parametrages globaux

Les hyperparametres reprennent la **Section 5.4** de l'article :

- taille d'embedding des labels d'emotion : `48` ;
- seuil de retention `beta = 0.1`, nombre d'emotions partagees `n_f = 3` ;
- triplets / entites injectes dans le prompt : `p_nt = 2`, `p_ne = 4` ;
- **feedback-aware reweighting** (Eq. 7) : `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- batch size `128` pour la recommandation, `16` pour la generation ;
- `lr = 1e-4` (DialoGPT) / `5e-5` (Llama 2-Chat + LoRA).

### Flags d'execution

| Flag | Defaut | Effet |
|---|---|---|
| `dry_run` | `True` | Court-circuite clone / archives / trainings / inference -> juste EDA + tables article. |
| `use_pretrained_ckpt` | `False` | Copie `data/ckpt/{pre,rec,emp}` dans `src_emo/data/saved/` et saute les 3 `train_*.py`. `infer_emp.py` est toujours lance. |
| `batch_scale` | `1.0` | Multiplicateur du `per_device_train_batch_size`. Le `gradient_accumulation_steps` est divise d'autant, donc l'**effective batch reste identique** a l'article. Valeur > 1.0 utile si GPU > 24 GB (ex. RTX 5090 32 GB -> `2.0` a tester). |
| `mixed_precision` | `"no"` | `"bf16"` recommande sur Blackwell / Ampere / Ada (≈ x1.5-2 wall-time, perte negligeable). `"fp16"` sinon. |

Les reglages **par defaut reproduisent fidelement l'article**. Les deux derniers flags sont des leviers hardware sans impact sur le LR ni la semantique du training.

In [3]:
# Cellule 3 - Parametres globaux du pipeline ECR
def get_config():
    return {
        # Chemins principaux
        "root": ROOT,
        "ecr_repo_url": "https://github.com/zxd-octopus/ECR.git",
        "ecr_dir": ROOT / "ECR",
        "src_emo_dir": ROOT / "ECR" / "src_emo",
        "emo_data_archive": ROOT / "data" / "emo_data.zip",
        "emo_data_dir": ROOT / "data" / "emo_data",
        "ckpt_archive": ROOT / "data" / "ckpt.zip",
        "ckpt_dir": ROOT / "data" / "ckpt",
        "results_dir": ROOT / "results",
        "generations_dir": ROOT / "results" / "generations",
        "figures_dir": ROOT / "results" / "figures",
        "logs_dir": ROOT / "logs",
        # Fichiers de metriques (optionnels : repli sur valeurs de l'article sinon)
        "objective_file": "objective_metrics.csv",
        "subjective_llm_file": "subjective_metrics_llm.csv",
        "subjective_human_file": "subjective_metrics_human.csv",
        "ablation_file": "ablation_metrics.csv",
        # Archives externes (Google Drive - publiees par les auteurs ECR)
        "emo_data_gdrive_id": "1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_",
        "ckpt_gdrive_id":     "1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19",
        # Hyperparametres article (Section 5.4)
        "emotion_embed_dim": 48,
        "beta": 0.1,
        "n_f": 3,
        "p_nt": 2,
        "p_ne": 4,
        "feedback_weights": {"like": 2.0, "dislike": 1.0, "not say": 0.5},
        "lr_dialogpt": 1e-4,
        "lr_llama": 5e-5,
        "batch_rec": 128,
        "batch_gen": 16,
        "seed": 42,
        # Flags d'execution
        "dry_run": False,
        "use_pretrained_ckpt": False,
        "skip_clone": False,
        "skip_download": False,
        # Optimisations hardware : par DEFAUT on reste FIDELE A L ARTICLE
        # (fp32, pas de torch.compile). Reactivable au cas par cas.
        "batch_scale": 1.0,
        "mixed_precision": "no",        # article fp32
        "num_workers": 16,
        "enable_torch_compile": False,  # article : pas de torch.compile
        # Overrides specifiques phase rec (laisses None => heritent des flags globaux)
        "rec_enable_torch_compile": False,
        "rec_mixed_precision": "no",
        "rec_batch_scale": 1.0,
        # Strategie export rec : "best" = valid loss, "final" = dernier epoch.
        # L'article sauvegarde les deux ; on prend best par defaut.
        "rec_export_checkpoint": "best",
        # Modeles de base requis par les scripts ECR
        "base_models": {
            "microsoft/DialoGPT-small": "save/dialogpt",
            "roberta-base":             "save/roberta",
        },
        # --------- Phase Llama / Qwen (Table 2 + Appendix E) ----------
        # Par defaut desactives : activable pour produire les 3 lignes reproductibles.
        "run_llama_zero_shot": True,
        "run_llama_lora": True,
        "run_llm_scorer": True,
        "run_kappa": True,
        # Backend generation Llama : "hf" (transformers pipeline) ou "vllm".
        "llama_backend": "vllm",
        "llama_vllm_autostart": True,
        "llama_serve_port": 8001,
        # Timeout vLLM ready : premier run = ~14 min (download HF) + 1-2 min
        # (compile CUDA graphs + warmup) ; runs suivants = ~10 s (cache chaud).
        # 1800 s couvre largement le cold start et permet un fallback HF sain
        # via la detection de proc.poll() (crash precoce sans attendre la fin).
        "llama_vllm_ready_timeout": 1800,
        # gpu_memory_utilization vLLM : voir note Cellule 9b. 0.85 est la valeur
        # recommandee par vLLM 0.19 pour Llama-2-7B bf16 sur 32 GB (poids
        # ~12.5 GB + CUDA graphs ~12 GB + KV cache ~2.5 GB).
        "llama_vllm_gmu": 0.85,
        # Le repo officiel `meta-llama/Llama-2-7b-chat-hf` est gated (licence
        # Meta + approbation manuelle). `NousResearch/Llama-2-7b-chat-hf` est
        # un miroir public des memes poids bit-a-bit : les resultats sont
        # identiques. Si vous etes approuve par Meta, repassez sur le repo
        # officiel (une seule ligne a changer, aucun autre impact).
        "llama_base_model": "NousResearch/Llama-2-7b-chat-hf",
        "llama_local_dir": None,
        "llama_lora_dir": ROOT / "data" / "lora_ecr_llama2_chat",
        # LoRA hyperparams (Section 5.4)
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lora_epochs": 15,
        "lora_per_device_batch": 4,
        "lora_grad_accum": 4,
        "lora_use_4bit": False,
        "use_flash_attn_2": True,
        # Scorer local Qwen2.5-32B-AWQ (remplacement GPT-4-turbo Appendix E)
        "scorer_model": "Qwen/Qwen2.5-32B-Instruct-AWQ",
        "scorer_port": 8000,
        "scorer_max_model_len": 8192,
        "scorer_sample_size": 200,
        "scorer_concurrency": 16,
        # Garde VRAM avant de lancer Qwen 32B AWQ. Sous ce seuil, on abort
        # explicitement plutot que de laisser vLLM OOM apres 5-10 min de
        # download + compile. 22 GB = poids AWQ Qwen (~20 GB) + marge CUDA
        # context/graph. Abaisser a 18.0 pour tenter le coup si fragmente.
        "scorer_min_free_gb": 22.0,
    }


def accelerate_launch_cmd(cfg, script):
    """Construit `accelerate launch [--mixed_precision X] script` selon la config."""
    cmd = ["accelerate", "launch"]
    mp = cfg.get("mixed_precision", "no")
    if mp and mp != "no":
        cmd += ["--mixed_precision", mp]
    cmd.append(script)
    return cmd


def accelerate_launch_cmd_phase(cfg, script, phase=None):
    """Construit `accelerate launch [--mixed_precision X] script` pour une phase donnee."""
    cmd = ["accelerate", "launch"]
    mp = cfg.get("mixed_precision", "no")
    if phase == "rec" and cfg.get("rec_mixed_precision") is not None:
        mp = cfg["rec_mixed_precision"]
    if mp and mp != "no":
        cmd += ["--mixed_precision", mp]
    cmd.append(script)
    return cmd


def scaled_batch_phase(cfg, per_device_batch, grad_accum, phase=None):
    """Scale batch en preservant l'effective batch. Override par phase supporte."""
    scale = float(cfg.get("batch_scale", 1.0))
    if phase == "rec" and cfg.get("rec_batch_scale") is not None:
        scale = float(cfg["rec_batch_scale"])
    scale = max(1.0, scale)
    if scale == 1.0:
        return per_device_batch, grad_accum
    effective = per_device_batch * grad_accum
    new_batch = max(1, int(round(per_device_batch * scale)))
    new_grad_accum = max(1, effective // new_batch)
    return new_batch, new_grad_accum


def scaled_batch(cfg, per_device_batch, grad_accum):
    """Ancien helper (compat) : equivalent a scaled_batch_phase sans phase."""
    return scaled_batch_phase(cfg, per_device_batch, grad_accum, phase=None)


cfg = get_config()
cfg


{'root': PosixPath('/workspace'),
 'ecr_repo_url': 'https://github.com/zxd-octopus/ECR.git',
 'ecr_dir': PosixPath('/workspace/ECR'),
 'src_emo_dir': PosixPath('/workspace/ECR/src_emo'),
 'emo_data_archive': PosixPath('/workspace/data/emo_data.zip'),
 'emo_data_dir': PosixPath('/workspace/data/emo_data'),
 'ckpt_archive': PosixPath('/workspace/data/ckpt.zip'),
 'ckpt_dir': PosixPath('/workspace/data/ckpt'),
 'results_dir': PosixPath('/workspace/results'),
 'generations_dir': PosixPath('/workspace/results/generations'),
 'figures_dir': PosixPath('/workspace/results/figures'),
 'logs_dir': PosixPath('/workspace/logs'),
 'objective_file': 'objective_metrics.csv',
 'subjective_llm_file': 'subjective_metrics_llm.csv',
 'subjective_human_file': 'subjective_metrics_human.csv',
 'ablation_file': 'ablation_metrics.csv',
 'emo_data_gdrive_id': '1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_',
 'ckpt_gdrive_id': '1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19',
 'emotion_embed_dim': 48,
 'beta': 0.1,
 'n_f': 3,
 'p_nt': 2

## 2. Preparation de l'environnement

Nous clonons le depot [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR) puis telechargeons les deux archives publiees par les auteurs sur Google Drive :

- [`emo_data.zip`](https://drive.google.com/file/d/1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_/view) (~111 MB) — donnees emotionnelles pretraitees (annotations GPT-3.5 + reviews IMDb filtrees — Section 4.1). Contient `llama_train.json`, `llama_test.json`, les entites DBpedia et les reviews retenues.
- [`ckpt.zip`](https://drive.google.com/file/d/1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19/view) (~679 MB) — poids entraines (GPT-2 classifieur d'emotions + ECR[DialoGPT] + ECR[Llama 2-Chat]).

Le telechargement utilise [`gdown`](https://github.com/wkentaro/gdown) qui gere automatiquement l'interstitiel Google Drive "virus scan warning" servi pour tout fichier > 100 MB. En cas d'echec reseau, deposer manuellement les deux `.zip` dans `data/` puis relancer la cellule.

In [4]:
# Cellule 4 - Clone du depot officiel ECR
def clone_ecr_repo(cfg):
    """Clone le depot zxd-octopus/ECR a la racine du projet (idempotent)."""
    if cfg["skip_clone"] or cfg["ecr_dir"].exists():
        print(f"ECR repo deja present: {cfg['ecr_dir']}")
        return cfg["ecr_dir"]
    print(f"Clonage de {cfg['ecr_repo_url']} -> {cfg['ecr_dir']}")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", cfg["ecr_repo_url"], str(cfg["ecr_dir"])],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    return cfg["ecr_dir"]

### 2bis. Patches de compatibilite stack moderne

Le code ECR a ete ecrit pour `transformers 4.15` / `pyg 2.0.1` (fin 2023). Plusieurs APIs ont bouge depuis :

| Symbole | Statut | Correctif |
|---|---|---|
| `transformers.AdamW` | Retire en 4.30+ | -> `torch.optim.AdamW` (impl fusee CUDA plus rapide) |
| `transformers.modeling_utils.find_pruneable_heads_and_indices` | Deplace 4.17 | -> `transformers.pytorch_utils` |
| `transformers.utils.model_parallel_utils` | Retire 4.40+ | Stubs no-op (GPT-2 `parallelize()` non utilise par ECR) |
| `pyg.typing.SparseTensor` | Requiert `torch-sparse` | Format standard Tensor[2,E] + edge_type separe (compat RGCNConv) |

`patch_ecr_compat(cfg)` applique 5 patches idempotents sur les fichiers clones dans `ECR/src_emo/`. Aucun PR upstream ni download externe requis.

In [5]:
# Cellule 4b - Compatibilite stack moderne (transformers >= 4.40 / pyg >= 2.5 / accelerate >= 0.20 / torch >= 2)
# ECR a ete ecrit pour transformers 4.15 / pyg 2.0.1 / accelerate 0.8 / torch 1.8.
# On applique des patches idempotents aux fichiers clones. Aucune PR upstream requise ;
# un `cd ECR && git checkout -- src_emo/` restaure l'original et patch_ecr_compat
# se re-applique au run suivant.
import re as _re


def _patch_file(path, replacements, description, skip_if=None):
    """Applique des remplacements regex idempotents.

    Args:
        skip_if: chaine. Si presente dans le fichier, on saute toute modification
                 (protection anti-reinsertion pour les patches additifs).
    """
    if not path.exists():
        print(f"[patch] SKIP {description}: {path} absent")
        return
    text = path.read_text()
    original = text
    if skip_if is not None and skip_if in text:
        print(f"[patch] {description}: deja applique (marker)")
        return
    for pat, repl in replacements:
        text = _re.sub(pat, repl, text, flags=_re.MULTILINE)
    if text == original:
        print(f"[patch] {description}: deja applique (idempotent)")
        return
    path.write_text(text)
    print(f"[patch] {description}: OK")


def _insert_block_after(path, anchor_regex, block, description, skip_if):
    """Insere `block` juste apres la premiere ligne matchant anchor_regex."""
    if not path.exists():
        print(f"[patch] SKIP {description}: {path} absent")
        return
    text = path.read_text()
    if skip_if in text:
        print(f"[patch] {description}: deja applique (marker)")
        return
    m = _re.search(anchor_regex, text, flags=_re.MULTILINE)
    if not m:
        print(f"[patch] SKIP {description}: anchor introuvable")
        return
    new_text = text[:m.end()] + "\n" + block.rstrip("\n") + "\n" + text[m.end():]
    path.write_text(new_text)
    print(f"[patch] {description}: OK")


def _insert_block_before_first(path, anchor_regex, block, description, skip_if):
    """Insere `block` juste avant la 1ere ligne matchant anchor_regex."""
    if not path.exists():
        print(f"[patch] SKIP {description}: {path} absent")
        return
    text = path.read_text()
    if skip_if in text:
        print(f"[patch] {description}: deja applique (marker)")
        return
    m = _re.search(anchor_regex, text, flags=_re.MULTILINE)
    if not m:
        print(f"[patch] SKIP {description}: anchor introuvable")
        return
    new_text = text[:m.start()] + block + text[m.start():]
    path.write_text(new_text)
    print(f"[patch] {description}: OK")


def _collapse_duplicated_block(path, block_regex, description, max_iter=30):
    """Remplace 2+ occurrences consecutives du bloc par une seule (idempotent)."""
    if not path.exists():
        return
    text = path.read_text()
    original = text
    # Repeter tant qu'on peut fusionner des paires contigues.
    for _ in range(max_iter):
        new_text = _re.sub(rf'({block_regex}){{2,}}', r'\1', text, flags=_re.MULTILINE)
        if new_text == text:
            break
        text = new_text
    if text != original:
        path.write_text(text)
        print(f"[patch] {description}: cleanup OK ({len(original)-len(text)} bytes)")




def patch_ecr_compat(cfg):
    """Rend le code ECR compatible avec transformers >= 4.40 et pyg >= 2.5.

    Patches appliques (tous idempotents) :
      1. `transformers.AdamW` retire -> `torch.optim.AdamW` (train_{pre,rec,emp}.py)
      2. `modeling_utils.find_pruneable_heads_and_indices` -> `pytorch_utils`
      3. `model_parallel_utils` retire en 4.40 -> stubs no-op (jamais appele)
      4. `SparseTensor` (pyg >= 2.5 exige torch-sparse) -> Tensor [2,E] + edge_type
      5. RGCNConv call dans model_prompt.py -> forme standard (x, edge_index, edge_type)
      6. `Accelerator(..., fp16=...)` retire en accelerate 0.12 -> drop kwarg (mixed_precision via `accelerate launch`)
      7. `torch.set_deterministic` retire en torch 2.0 -> `torch.use_deterministic_algorithms(True, warn_only=True)`
         + CUDA_LAUNCH_BLOCKING / CUBLAS_WORKSPACE_CONFIG commentes (serialisation kernels -> 5-10x plus lent)
      8. `accelerator.use_fp16` retire en accelerate 0.20 -> getattr fallback (bf16 via launch)
      9. `.cuda()` hardcode dans dataset_dbpedia.py -> `.to(device)` pour tolerer CPU transitoire
     10. `print("here")` debug de l'auteur supprime (5 locations : train_pre/dataset_emp/imdb_filter/co_appear)
     11. `evaluate_rec.py` cree `save/redial_rec/` avant ecriture de `rec.json`
     12. `transformers.file_utils.ModelOutput` deprecated 4.25 (retire 5.0) -> `transformers.utils.ModelOutput`
     13. `PromptGPT2forCRS` herite aussi de `GenerationMixin` (transformers >= 4.50)
     14. `train_rec.py` ne force plus `CUDA_VISIBLE_DEVICES=3`
     15. `log/` est cree avant `logger.add(...)` dans train_{pre,rec,emp}.py + infer_emp.py
     16. chemins de sortie stabilises (`pre`, `rec`, `emp`) sans suffixe timestamp
     17. `train_pre.py` sauvegarde par defaut dans `data/saved/pre`
     18. `train_emp.py` sauvegarde par defaut dans `data/saved/emp`
     19. `infer_emp.py` charge localement (`local_files_only=True`) si le path modele existe
     20. suppression de `retain_graph=True` dans train_pre/train_rec
     29. `train_emp.py` save_pretrained avec safe_serialization=False (tied weights GPT-2)
     30. `from collections.abc import Mapping` (train/infer pour move_batch_to_device)
     31. `--num_workers default=16` (au lieu de 0) dans les 4 scripts CLI
     32. `--enable_torch_compile` flag CLI (opt-in torch.compile)
     33. helper `move_batch_to_device(batch, device)` (gere Tensor/Mapping/list/tuple + BatchEncoding)
     34. bloc `torch.compile(model)` gate sur --enable_torch_compile (train_pre/rec/emp)
     35. DataLoader tuning : `num_workers` + `pin_memory` + `persistent_workers`
     36. `batch = move_batch_to_device(batch, device)` dans chaque boucle train/valid/test
     37. `collate_device = torch.device("cpu")` + substitution `device=self.device` dans dataset_*.py
         (evite "Cannot re-initialize CUDA in forked subprocess" avec num_workers>0)

    De-duplications automatiques au demarrage (fix anciennes inidempotences Patches 11/25/27):
     - `os.makedirs("log", ...)` : collapse 2+ copies en 1
     - `import os\nos.makedirs("save/redial_rec", ...)` : collapse 2+ copies en 1
     - `if os.path.isdir(args.model)` : remise a plat de la recursion imbriquee
    """
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo absent ({src_emo}) -> patches ignores.")
        return

    # -- Cleanup preambule : collapse les duplications heritees de Patches 11/25/27 --
    # Les anciennes versions des Patches 11, 25, 27 etaient non-idempotentes : chaque
    # reapplication ajoutait un bloc supplementaire. On collapse les copies avant toute chose.
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _collapse_duplicated_block(
            src_emo / name,
            r'(?:    os\.makedirs\("log", exist_ok=True\)\n)',
            description=f"cleanup dup log/ ({name})",
        )
    _collapse_duplicated_block(
        src_emo / "evaluate_rec.py",
        r'(?:        import os\n\n        os\.makedirs\("save/redial_rec", exist_ok=True\)  # PATCH: cree le dossier de logs\n\n)',
        description="cleanup dup save/redial_rec (evaluate_rec.py)",
    )
    # Patch 27 : recursion imbriquee "if os.path.isdir(args.model):" ... a remettre a plat.
    _patch_file(src_emo / "infer_emp.py", [(
        r'(^(\s*)if os\.path\.isdir\(args\.model\):\n\s+model = PromptGPT2forCRS\.from_pretrained\(args\.model, local_files_only=True\)\n\s+else:\n)(?:\s+if os\.path\.isdir\(args\.model\):\n\s+model = PromptGPT2forCRS\.from_pretrained\(args\.model, local_files_only=True\)\n\s+else:\n)+(\s+model = PromptGPT2forCRS\.from_pretrained\(args\.model\))',
        r'\2if os.path.isdir(args.model):\n\2    model = PromptGPT2forCRS.from_pretrained(args.model, local_files_only=True)\n\2else:\n\2    model = PromptGPT2forCRS.from_pretrained(args.model)',
    )], description="infer_emp cleanup nested isdir", skip_if=None)

    # Patch 1 : AdamW dans les 3 scripts de training
    adamw_patch = [(
        r"from transformers import AdamW, get_linear_schedule_with_warmup, AutoTokenizer, AutoModel",
        "from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel\nfrom torch.optim import AdamW",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py"):
        _patch_file(src_emo / name, adamw_patch, description=f"AdamW -> torch.optim ({name})")

    # Patches 2 + 3 : model_gpt2.py
    _patch_file(src_emo / "model_gpt2.py", [
        (
            r"from transformers\.modeling_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
            "from transformers.pytorch_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
        ),
        (
            r"^from transformers\.utils\.model_parallel_utils import assert_device_map, get_device_map\s*$",
            (
                "try:\n"
                "    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\n"
                "except ImportError:  # transformers >= 4.40 a retire le parallelisme custom GPT-2\n"
                "    def assert_device_map(*a, **kw): return None\n"
                "    def get_device_map(*a, **kw): return {}\n"
            ),
        ),
    ], description="model_gpt2.py imports")

    # Patch 4 : SparseTensor dans dataset_dbpedia.py
    _patch_file(src_emo / "dataset_dbpedia.py", [(
        r"^(\s*)self\.edge_index = SparseTensor\(row=self\.edge_index\[0\], col=self\.edge_index\[1\], value=self\.edge_type\)\s*$",
        r"\1# PATCH: SparseTensor supprime (PyG >= 2.5 requiert torch-sparse).\n"
        r"\1# edge_index reste Tensor [2,E], edge_type reste Tensor [E] -> format PyG standard.\n"
        r"\1# (voir model_prompt.py pour l'appel RGCNConv correspondant)",
    )], description="dataset_dbpedia.py SparseTensor")

    # Patch 5 : RGCNConv call dans model_prompt.py - on passe a la forme standard (3 args)
    _patch_file(src_emo / "model_prompt.py", [(
        r"entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index\) \+ node_embeds.*?\n\s*#entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index, self\.edge_type\) \+ node_embeds",
        "entity_embeds = self.kg_encoder(node_embeds, self.edge_index, self.edge_type) + node_embeds",
    )], description="model_prompt.py RGCNConv call")

    # Patch 6 : Accelerator(fp16=...) retire en accelerate 0.12+
    #           On s'appuie sur `accelerate launch --mixed_precision <mp>` pour la config.
    accel_patch = [(
        r"accelerator = Accelerator\(device_placement=False, fp16=args\.fp16\)",
        "accelerator = Accelerator(device_placement=False)  # PATCH: mixed_precision via `accelerate launch`",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, accel_patch, description=f"Accelerator fp16 kwarg ({name})")

    # Patch 7 : seed_torch dans config.py
    #   - torch.set_deterministic -> torch.use_deterministic_algorithms (retire en torch 2.0)
    #   - CUDA_LAUNCH_BLOCKING=1 et CUBLAS_WORKSPACE_CONFIG:4096:8 : residus debug qui
    #     serialisent les kernels CUDA et coutent 5-10x en wall-time sur GPU moderne.
    _patch_file(src_emo / "config.py", [
        (
            r"torch\.set_deterministic\(True\)",
            "torch.use_deterministic_algorithms(True, warn_only=True)  # PATCH: set_deterministic retire en torch 2.0",
        ),
        (
            r'^(\s*)os\.environ\["CUDA_LAUNCH_BLOCKING"\] = "1"\s*$',
            r'\1# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # PATCH: desactive (5-10x plus lent sur GPU moderne)',
        ),
        (
            r'^(\s*)os\.environ\["CUBLAS_WORKSPACE_CONFIG"\] = ":4096:8"\s*$',
            r'\1# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # PATCH: desactive (lie a determinisme strict)',
        ),
    ], description="config.py seed_torch")

    # Patch 8 : accelerator.use_fp16 retire en accelerate 0.20
    #   Les collators lisent ce flag pour activer `torch.cuda.amp.autocast` manuellement ;
    #   comme on passe --mixed_precision bf16 via `accelerate launch`, l'autocast est deja
    #   gere et on peut renvoyer False (fallback sur bf16 automatique).
    use_fp16_patch = [(
        r"accelerator\.use_fp16",
        "getattr(accelerator, 'use_fp16', False)",
    )]
    for name in ("train_pre.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, use_fp16_patch, description=f"accelerator.use_fp16 ({name})")

    # Patch 9 : .cuda() hardcode dans dataset_dbpedia.py
    #   Tolere un fallback CPU si aucun GPU n'est detecte au moment de construire DBpedia
    #   (cas transitoire : process zombie tenant de la VRAM, driver en power-save, ...).
    #   RGCNConv gere ensuite le deplacement vers le device du modele lors du forward.
    _patch_file(src_emo / "dataset_dbpedia.py", [
        (
            r"^([ \t]+)self\.edge_index = edge\[:, :2\]\.t\(\)\.cuda\(\)$",
            r"\1_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n\1self.edge_index = edge[:, :2].t().to(_dev)  # PATCH: tolere CPU",
        ),
        (
            r"^([ \t]+)self\.edge_type = edge\[:, 2\]\.cuda\(\)$",
            r"\1self.edge_type = edge[:, 2].to(_dev)  # PATCH: tolere CPU",
        ),
    ], description="dataset_dbpedia.py .cuda() hardcode")

    # Patch 10 : prints de debug `print("here")` laisses par les auteurs
    #   dataset_emp.py (2 occurrences), imdb_review_entity_filter.py, train_pre.py, co_appear.py
    debug_patch = [(
        r"^(\s*)print\(([\"'])here\2\)\s*$",
        r'\1pass  # PATCH: debug print("here") supprime',
    )]
    for name in ("dataset_emp.py", "imdb_review_entity_filter.py", "train_pre.py", "co_appear.py"):
        _patch_file(src_emo / name, debug_patch, description=f'debug print("here") dans {name}')

    # Patch 11 : ensure save/redial_rec existe avant ecriture rec.json
    _patch_file(src_emo / "evaluate_rec.py", [(
        r'^(\s*)self\.log_file = open\("save/redial_rec/rec\.json", ["\']w["\'], buffering=1\)\s*$',
        r'\1import os\n\1os.makedirs("save/redial_rec", exist_ok=True)  # PATCH: cree le dossier de logs\n\1self.log_file = open("save/redial_rec/rec.json", "w", buffering=1)',
    )], description="evaluate_rec.py rec.json directory",
       skip_if='os.makedirs("save/redial_rec", exist_ok=True)')

    # Patch 12 : transformers.file_utils -> transformers.utils (preemptif)
    #   file_utils est un alias deprecie depuis 4.25 et retire en 5.0.
    #   Ca marche encore en 4.57 mais emet un FutureWarning.
    _patch_file(src_emo / "model_gpt2.py", [(
        r"from transformers\.file_utils import ModelOutput",
        "from transformers.utils import ModelOutput  # PATCH: file_utils deprecated -> utils",
    )], description="file_utils -> utils (model_gpt2.py)")
    # Patch 21 : train_rec.py charge par defaut le checkpoint pre/final
    _patch_file(src_emo / "train_rec.py", [(
        r'parser\.add_argument\("--prompt_encoder", type=str,default="data/saved/pre/"\)',
        'parser.add_argument("--prompt_encoder", type=str,default="data/saved/pre/final/")',
    )], description="train_rec.py prompt_encoder default")

    # Patch 22 : merge_rec.py explicite les cas rec.json vide/incomplet
    _patch_file(src_emo / "data" / "redial_gen" / "merge_rec.py", [
        (
            r'gen_data = gen_file\.readlines\(\)',
            (
                "gen_data = gen_file.readlines()\n"
                "if len(gen_data) == 0:\n"
                "    raise RuntimeError(\n"
                "        f\"{gen_file_path} est vide: train_rec n'a pas produit de predictions.\"\n"
                "    )"
            ),
        ),
        (
            r"for i in range\(len\(raw\['rec'\]\)\):\n\s*gen = json\.loads\(gen_data\[cnt\]\)",
            (
                "for i in range(len(raw['rec'])):\n"
                "                if cnt >= len(gen_data):\n"
                "                    raise RuntimeError(\n"
                "                        f\"Nombre de predictions insuffisant dans {gen_file_path}: \"\n"
                "                        f\"{len(gen_data)} lignes, au moins {cnt + 1} requises.\"\n"
                "                    )\n"
                "                gen = json.loads(gen_data[cnt])"
            ),
        ),
    ], description="merge_rec.py guardrails",
       skip_if="est vide: train_rec n'a pas produit de predictions.")


    # Patch 23 : PromptGPT2forCRS herite de GenerationMixin (transformers >= 4.50)
    _patch_file(src_emo / "model_gpt2.py", [
        (
            r'^from transformers import Conv1D\s*$',
            "from transformers import Conv1D\nfrom transformers.generation import GenerationMixin",
        ),
        (
            r'^class PromptGPT2forCRS\(GPT2PreTrainedModel\):\s*$',
            "class PromptGPT2forCRS(GPT2PreTrainedModel, GenerationMixin):",
        ),
    ], description="model_gpt2.py GenerationMixin",
       skip_if="from transformers.generation import GenerationMixin")

    # Patch 24 : train_rec.py n'impose pas CUDA_VISIBLE_DEVICES=3
    _patch_file(src_emo / "train_rec.py", [(
        r'^os\.environ\["CUDA_VISIBLE_DEVICES"\] = "3"\s*$',
        "# PATCH: suppression CUDA_VISIBLE_DEVICES hardcode",
    )], description="train_rec.py CUDA_VISIBLE_DEVICES")

    # Patch 25 : dossiers log/ assures avant logger.add
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, [(
            r'^(\s*)local_time = time\.strftime\("%Y-%m-%d-%H-%M-%S", time\.localtime\(\)\)\s*$',
            r'\1local_time = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())\n\1os.makedirs("log", exist_ok=True)',
        )], description=f"{name} log directory",
           skip_if='os.makedirs("log", exist_ok=True)')

    # Patch 26 : sorties stables (sans suffixe timestamp) pour chainage pre->rec et emp->infer
    _patch_file(src_emo / "train_pre.py", [
        (
            r'parser\.add_argument\("--output_dir", type=str, default=\'data/saved/pre-trained\'',
            "parser.add_argument(\"--output_dir\", type=str, default=\'data/saved/pre\'",
        ),
        (
            r'^(\s*)args\.output_dir = args\.output_dir \+ local_time\s*$',
            r'\1# PATCH: keep stable output dir (no timestamp)',
        ),
    ], description="train_pre.py stable output_dir")

    _patch_file(src_emo / "train_emp.py", [
        (
            r'parser\.add_argument\("--output_dir", type=str, help="Where to store the final model\.",default="data/saved/emp_conv"\)',
            'parser.add_argument("--output_dir", type=str, help="Where to store the final model.", default="data/saved/emp")',
        ),
        (
            r'^(\s*)args\.output_dir = args\.output_dir\+local_time\s*$',
            r'\1# PATCH: keep stable output dir (no timestamp)',
        ),
    ], description="train_emp.py stable output_dir")

    _patch_file(src_emo / "train_rec.py", [(
        r'^(\s*)args\.output_dir = args\.output_dir \+ local_time\s*$',
        r'\1# PATCH: keep stable output dir (no timestamp)',
    )], description="train_rec.py stable output_dir")

    # Patch 27 : infer_emp.py charge local_files_only si le modele est un dossier local
    _patch_file(src_emo / "infer_emp.py", [(
        r'^(\s*)model = PromptGPT2forCRS\.from_pretrained\(args\.model\)\s*$',
        r'\1if os.path.isdir(args.model):\n\1    model = PromptGPT2forCRS.from_pretrained(args.model, local_files_only=True)\n\1else:\n\1    model = PromptGPT2forCRS.from_pretrained(args.model)',
    )], description="infer_emp.py local model loading",
       skip_if='if os.path.isdir(args.model):\n        model = PromptGPT2forCRS.from_pretrained(args.model, local_files_only=True)')

    # Patch 28 : suppress retain_graph=True (memoire inutile)
    for name in ("train_pre.py", "train_rec.py"):
        _patch_file(src_emo / name, [(
            r'accelerator\.backward\(loss, retain_graph = True\)',
            'accelerator.backward(loss)',
        )], description=f"{name} retain_graph")

    # Patch 29 : train_emp.py save_pretrained -> safe_serialization=False
    # GPT-2 tie les poids lm_head.weight et transformer.wte.weight; safetensors refuse ce tying.
    _patch_file(src_emo / "train_emp.py", [
        (
            r'model\.save_pretrained\(best_metric_dir\)',
            'model.save_pretrained(best_metric_dir, safe_serialization=False)',
        ),
        (
            r'model\.save_pretrained\(args\.output_dir\)',
            'model.save_pretrained(args.output_dir, safe_serialization=False)',
        ),
    ], description="train_emp.py save_pretrained safe_serialization=False")
    # =============================================================
    # Patches modernisation (30-36) : reproductibilite end-to-end
    # Ces modifications etaient manquantes de patch_ecr_compat et
    # existaient uniquement dans le checkout local ECR/.
    # =============================================================

    TRAIN_SCRIPTS = ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py")

    # Patch 30 : import `from collections.abc import Mapping`
    for name in TRAIN_SCRIPTS:
        _patch_file(src_emo / name, [(
            r'^import time\s*$',
            'import time\nfrom collections.abc import Mapping',
        )], description=f"Mapping import ({name})",
           skip_if="from collections.abc import Mapping")

    # Patch 31 : --num_workers default=16 (article defaut=0 mais on veut 16 dans le CLI)
    for name in TRAIN_SCRIPTS:
        _patch_file(src_emo / name, [(
            r"parser\.add_argument\('--num_workers', type=int, default=0\)",
            "parser.add_argument('--num_workers', type=int, default=16)",
        )], description=f"num_workers default=16 ({name})")

    # Patch 32 : --enable_torch_compile CLI flag
    for name in TRAIN_SCRIPTS:
        _patch_file(src_emo / name, [(
            r'^(\s*)args = parser\.parse_args\(\)',
            r'\1parser.add_argument("--enable_torch_compile", action="store_true", default=False)\n\1args = parser.parse_args()',
        )], description=f"--enable_torch_compile arg ({name})",
           skip_if='"--enable_torch_compile"')

    # Patch 33 : helper `move_batch_to_device` pour transferer Tensor/Mapping/... vers CUDA
    move_helper = (
        "def move_batch_to_device(batch, device):\n"
        "    if torch.is_tensor(batch):\n"
        "        return batch.to(device, non_blocking=True)\n"
        "    if hasattr(batch, \"to\"):\n"
        "        try:\n"
        "            return batch.to(device)\n"
        "        except Exception:\n"
        "            pass\n"
        "    if isinstance(batch, Mapping):\n"
        "        return {k: move_batch_to_device(v, device) for k, v in batch.items()}\n"
        "    if isinstance(batch, list):\n"
        "        return [move_batch_to_device(v, device) for v in batch]\n"
        "    if isinstance(batch, tuple):\n"
        "        return tuple(move_batch_to_device(v, device) for v in batch)\n"
        "    return batch\n"
    )
    for name in TRAIN_SCRIPTS:
        _insert_block_after(
            src_emo / name,
            r'^    return args\s*$',
            move_helper,
            description=f"move_batch_to_device ({name})",
            skip_if="def move_batch_to_device",
        )

    # Patch 34 : bloc torch.compile optionnel apres `model = model.to(device)`
    for name, label in (("train_pre.py", "train_pre"), ("train_rec.py", "train_rec"), ("train_emp.py", "train_emp")):
        compile_block = (
            "    if args.enable_torch_compile and hasattr(torch, \"compile\"):\n"
            "        try:\n"
            "            model = torch.compile(model, mode=\"reduce-overhead\")\n"
            f"            logger.info(\"torch.compile active ({label})\")\n"
            "        except Exception as e:\n"
            f"            logger.warning(f\"torch.compile indisponible ({label}): {{e}}\")\n"
        )
        _insert_block_after(
            src_emo / name,
            r'^    model = model\.to\(device\)\s*$',
            compile_block,
            description=f"torch.compile block ({name})",
            skip_if="if args.enable_torch_compile and hasattr(torch",
        )

    # Patch 35 : DataLoader tuning (num_workers + pin_memory + persistent_workers)
    # 35a : declarer loader_* UNE seule fois avant le premier DataLoader.
    loader_decl_block = (
        "    loader_num_workers = max(0, int(args.num_workers))\n"
        "    loader_pin_memory = torch.cuda.is_available()\n"
        "    loader_persistent_workers = loader_num_workers > 0\n"
    )
    for name in TRAIN_SCRIPTS:
        _insert_block_before_first(
            src_emo / name,
            r'^    (?:[a-z_]+_)?dataloader = DataLoader\(',
            loader_decl_block,
            description=f"DataLoader loader_* decl ({name})",
            skip_if="loader_num_workers = max(0, int(args.num_workers))",
        )

    # 35b : kwargs sur chaque DataLoader. 3 cas :
    #   A. shuffle=True sans virgule finale (train_dataloader dans train_pre/rec)
    #   B. collate_fn=... sans num_workers (valid/test dans train_pre/rec)
    #   C. num_workers=args.num_workers (train_emp, infer_emp) -> remplacement simple
    dl_shuffle_pat = (
        r'(^    [a-z_]+dataloader = DataLoader\(\n'
        r'        [a-z_]+(?:_dataset)?,\n'
        r'(?:        [^\n]+\n){0,5}'
        r'        shuffle=True)\n    \)'
    )
    dl_shuffle_repl = (
        r'\1,\n'
        '        num_workers=loader_num_workers,\n'
        '        pin_memory=loader_pin_memory,\n'
        '        persistent_workers=loader_persistent_workers,\n'
        '    )'
    )
    dl_noshuffle_pat = (
        r'(^    [a-z_]+dataloader = DataLoader\(\n'
        r'        [a-z_]+(?:_dataset|dataset),\n'
        r'        batch_size=args\.per_device_[a-z_]+_batch_size,\n'
        r'        collate_fn=data_collator(?:_[a-z_]+)?),\n    \)'
    )
    dl_noshuffle_repl = (
        r'\1,\n'
        '        num_workers=loader_num_workers,\n'
        '        pin_memory=loader_pin_memory,\n'
        '        persistent_workers=loader_persistent_workers,\n'
        '    )'
    )
    dl_has_workers_pat = r'(^        )num_workers=args\.num_workers,\s*$'
    dl_has_workers_repl = (
        r'\1num_workers=loader_num_workers,\n'
        '        pin_memory=loader_pin_memory,\n'
        '        persistent_workers=loader_persistent_workers,'
    )
    for name in TRAIN_SCRIPTS:
        _patch_file(src_emo / name, [
            (dl_shuffle_pat, dl_shuffle_repl),
            (dl_noshuffle_pat, dl_noshuffle_repl),
            (dl_has_workers_pat, dl_has_workers_repl),
        ], description=f"DataLoader kwargs tuning ({name})")

    # Patch 36 : `batch = move_batch_to_device(batch, device)` au debut de chaque
    #            boucle `for ... batch ... in ...dataloader`. Le helper gere ensuite
    #            Tensor/BatchEncoding/dict/list.
    loop_patches = [
        (
            r'^(        for step, batch in enumerate\(train_dataloader\):\s*$)\n(?!            batch = move_batch_to_device)',
            r'\1\n            batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(            for step, batch in enumerate\(train_dataloader\):\s*$)\n(?!                batch = move_batch_to_device)',
            r'\1\n                batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(        for batch in tqdm\(valid_dataloader\):\s*$)\n(?!            batch = move_batch_to_device)',
            r'\1\n            batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(        for batch in tqdm\(test_dataloader\):\s*$)\n(?!            batch = move_batch_to_device)',
            r'\1\n            batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(            for batch in tqdm\(valid_dataloader\):\s*$)\n(?!                batch = move_batch_to_device)',
            r'\1\n                batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(            for batch in tqdm\(test_dataloader\):\s*$)\n(?!                batch = move_batch_to_device)',
            r'\1\n                batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(        for batch in tqdm\(valid_dataloader, disable=not accelerator\.is_local_main_process\):\s*$)\n(?!            batch = move_batch_to_device)',
            r'\1\n            batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(            for batch in tqdm\(valid_dataloader, disable=not accelerator\.is_local_main_process\):\s*$)\n(?!                batch = move_batch_to_device)',
            r'\1\n                batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(        for batch in tqdm\(valid_gen_dataloader, disable=not accelerator\.is_local_main_process\):\s*$)\n(?!            batch = move_batch_to_device)',
            r'\1\n            batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(            for batch in tqdm\(valid_gen_dataloader, disable=not accelerator\.is_local_main_process\):\s*$)\n(?!                batch = move_batch_to_device)',
            r'\1\n                batch = move_batch_to_device(batch, device)\n',
        ),
        (
            r'^(    for batch in tqdm\(dataloader, disable=not accelerator\.is_local_main_process\):\s*$)\n(?!        batch = move_batch_to_device)',
            r'\1\n        batch = move_batch_to_device(batch, device)\n',
        ),
    ]
    for name in TRAIN_SCRIPTS:
        _patch_file(src_emo / name, loop_patches,
                    description=f"move_batch_to_device dans les boucles ({name})")

    # Patch 37 : dataset_*.py -> collate_device CPU pour eviter fork CUDA avec num_workers>0
    for name in ("dataset_pre.py", "dataset_rec.py", "dataset_emp.py"):
        _patch_file(src_emo / name, [
            # Declare self.collate_device juste apres self.device = device
            (
                r'^(\s+)self\.device = device\s*$',
                r'\1self.device = device\n\1self.collate_device = torch.device("cpu")',
            ),
            # Remplace device=self.device / device = self.device par device=self.collate_device
            (r'device=self\.device\b', 'device=self.collate_device'),
            (r'device = self\.device\b', 'device = self.collate_device'),
            # to(self.device) -> to(self.collate_device)
            (r'\.to\(self\.device\)', '.to(self.collate_device)'),
        ], description=f"collate_device CPU ({name})",
           skip_if='self.collate_device = torch.device("cpu")')



In [6]:
# Cellule 5 - Telechargement des archives emo_data / ckpt depuis Google Drive
# gdown est necessaire : Google Drive sert une page HTML d'interstitiel pour
# tout fichier > 100 MB (ckpt.zip pese 679 MB). curl recupererait alors le
# HTML au lieu du .zip. gdown detecte cette page et suit le token `confirm`.
import gdown


def download_gdrive(file_id, dst, description):
    """Telecharge une archive depuis Google Drive (gere l'interstitiel > 100 MB)."""
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 1024:
        print(f"[{description}] deja present: {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[{description}] gdown download (id={file_id}) -> {dst}")
    try:
        out = gdown.download(id=file_id, output=str(dst), quiet=False)
    except Exception as exc:
        print(f"[{description}] ECHEC gdown: {exc}")
        return None
    if out is None or not dst.exists() or dst.stat().st_size < 1024:
        print(f"[{description}] telechargement vide, deposer manuellement dans: {dst}")
        if dst.exists():
            dst.unlink(missing_ok=True)
        return None
    print(f"[{description}] OK ({dst.stat().st_size / 1e6:.1f} MB)")
    return dst


def unzip_if_needed(archive, dst_dir):
    """Decompresse l'archive si necessaire."""
    dst_dir = Path(dst_dir)
    if dst_dir.exists() and any(dst_dir.iterdir()):
        print(f"  {dst_dir} deja extrait.")
        return dst_dir
    if archive is None or not Path(archive).exists():
        print(f"  archive manquante, extraction ignoree.")
        return None
    dst_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["unzip", "-oq", str(archive), "-d", str(dst_dir)],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    print(f"  extrait -> {dst_dir}")
    return dst_dir


def download_external_assets(cfg):
    """Telecharge + extrait emo_data.zip et ckpt.zip depuis Google Drive."""
    if cfg["skip_download"]:
        print("skip_download=True -> archives non recuperees.")
        return
    download_gdrive(cfg["emo_data_gdrive_id"], cfg["emo_data_archive"], "emo_data")
    unzip_if_needed(cfg["emo_data_archive"], cfg["emo_data_dir"])
    download_gdrive(cfg["ckpt_gdrive_id"], cfg["ckpt_archive"], "ckpt")
    unzip_if_needed(cfg["ckpt_archive"], cfg["ckpt_dir"])

### 2ter. Modeles de base HF + poids pre-entraines (optionnel)

Les scripts ECR chargent DialoGPT-small et RoBERTa-base via des chemins **locaux** (`save/dialogpt/`, `save/roberta/`) au sein de `src_emo/`. On les telecharge une fois via `huggingface_hub.snapshot_download` — les caches Docker Compose (`hf_cache`) evitent tout re-download entre runs.

`install_pretrained_ckpts` est active **uniquement** si le flag `use_pretrained_ckpt=True`. Il copie :

- `data/ckpt/pre/`  -> `src_emo/data/saved/pre/`   (Emotional Semantic Fusion)
- `data/ckpt/rec/`  -> `src_emo/data/saved/rec/`   (Emotion-aware Recommendation)
- `data/ckpt/emp/`  -> `src_emo/data/saved/emp/`   (Emotion-aligned DialoGPT)

Ainsi `train_rec.py` (qui lit `--prompt_encoder data/saved/pre/`) et `infer_emp.py` (qui lit `--model data/saved/emp/`) trouvent les poids directement. Les trois cellules `run_*` respectent ce flag : elles sautent proprement le training correspondant. Passer `use_pretrained_ckpt=False` relance le pipeline complet sans code a modifier.

In [7]:
# Cellule 5b - Modeles de base HF + poids pre-entraines (optionnel)
# Les scripts ECR appellent `AutoTokenizer.from_pretrained("save/dialogpt/")` et
# `AutoModel.from_pretrained("save/roberta/")` : ces repertoires doivent exister
# dans src_emo/. On telecharge depuis HF Hub une fois (cache volume HF partage).
from huggingface_hub import snapshot_download


def install_base_models(cfg):
    """Telecharge DialoGPT-small + RoBERTa-base dans src_emo/save/ (HF snapshots)."""
    src_emo = cfg["src_emo_dir"]
    for hf_id, rel_dir in cfg["base_models"].items():
        local = src_emo / rel_dir
        if (local / "config.json").exists():
            print(f"[base] {hf_id} deja present: {local}")
            continue
        local.mkdir(parents=True, exist_ok=True)
        print(f"[base] snapshot_download {hf_id} -> {local}")
        try:
            snapshot_download(
                repo_id=hf_id,
                local_dir=str(local),
                local_dir_use_symlinks=False,
            )
        except TypeError:
            # API huggingface_hub >= 0.23 a retire local_dir_use_symlinks
            snapshot_download(repo_id=hf_id, local_dir=str(local))
        print(f"[base] {hf_id} OK ({sum(1 for _ in local.iterdir())} fichiers)")


def install_pretrained_ckpts(cfg):
    """Copie data/ckpt/{pre,rec,emp}/* dans src_emo/data/saved/{pre,rec,emp}/.

    Active seulement si cfg["use_pretrained_ckpt"] est True.
    Permet aux scripts train_rec.py (qui lit --prompt_encoder data/saved/pre/)
    et infer_emp.py (qui lit --model data/saved/emp/) de trouver les poids sans
    re-entrainement. Aucun fichier n'est supprime : on peut toujours repartir
    sur un training from-scratch en desactivant le flag.
    """
    if not cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=False -> poids fournis non installes (training complet).")
        return
    src_emo = cfg["src_emo_dir"]
    ckpt = cfg["ckpt_dir"]
    saved = src_emo / "data" / "saved"
    saved.mkdir(parents=True, exist_ok=True)
    for sub in ("pre", "rec", "emp"):
        src = ckpt / sub
        dst = saved / sub
        if not src.exists():
            print(f"[ckpt] source absente: {src}")
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.name.startswith(".") or f.name == "__MACOSX":
                continue
            target = dst / f.name
            if target.exists():
                continue
            if f.is_file():
                shutil.copy2(f, target)
            else:
                shutil.copytree(f, target)
        print(f"[ckpt] {src} -> {dst}")

## 3. Pretraitement du dataset ReDial (Section 4.1)

Le README du depot ECR enchaine les etapes suivantes :

```bash
cd src_emo
cp -r data/emo_data/* data/redial/
python data/redial/process.py
```

`process.py` integre les annotations GPT-3.5 (9 labels d'emotion : *like, curious, happy, grateful, negative, neutral, nostalgia, agreement, surprise* — cf. Section 4.1.1) au corpus ReDial brut et fournit les entites DBpedia alignees. Pour la recommandation, `process_mask.py` masque l'item cible ; `merge.py` recolte les predictions du modele conversationnel UniCRS. La fonction ci-dessous orchestre ces commandes.

In [8]:
# Cellule 6 - Pretraitement du dataset ReDial (Section 4.1)
def _run(cmd, cwd=None, description=""):
    """Wrapper subprocess avec affichage compact."""
    print(f"[run] {description} :: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        print("[STDERR]", result.stderr.strip())
    return result.returncode == 0


def prepare_redial_data(cfg):
    """Reproduit la preparation `data/redial/` et `data/redial_gen/` du README ECR."""
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo indisponible: {src_emo} -> clone le depot d'abord.")
        return False
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"
    emo = cfg["emo_data_dir"]

    redial.mkdir(parents=True, exist_ok=True)
    redial_gen.mkdir(parents=True, exist_ok=True)

    if emo.exists():
        for p in emo.iterdir():
            tgt = redial / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    else:
        print(f"emo_data non trouve ({emo}) - les scripts supposent qu'il a ete decompresse.")

    ok = True
    if cfg["dry_run"]:
        print("DRY_RUN=True -> scripts process.py / process_mask.py / merge.py non executes.")
        return True

    # Les scripts ECR utilisent des chemins relatifs (ex: open('entity2id.json')) :
    # ils doivent etre lances depuis leur propre repertoire, pas depuis src_emo.
    ok &= _run(["python", "process.py"],      cwd=redial,     description="process.py")
    ok &= _run(["python", "process_mask.py"], cwd=redial,     description="process_mask.py")
    if redial.exists() and redial_gen.exists():
        for p in redial.iterdir():
            tgt = redial_gen / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    ok &= _run(["python", "merge.py"],        cwd=redial_gen, description="merge.py")
    return ok

## 4. Sous-tache Emotional Semantic Fusion (Section 4.1)

La fusion emotionnelle est entrainee avec `train_pre.py`. L'objectif est de fusionner la semantique des mots (token-level) et des entites emotionnelles (utterance-level) avant la phase de recommandation. Les hyperparametres du depot sont :

- `num_train_epochs=10`, `gradient_accumulation_steps=4`, `per_device_train_batch_size=16` ;
- `max_length=200`, `prompt_max_length=200`, `entity_max_length=32` ;
- `learning_rate=5e-4`, `num_warmup_steps=1389`.

L'option `--nei_mer` active la fusion emotion/neighbour decrite Section 4.1.

In [9]:
# Cellule 7 - Emotional Semantic Fusion (train_pre.py)
def run_emotional_semantic_fusion(cfg):
    """Execute `accelerate launch train_pre.py ...` (Section 4.1)."""
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_pre.py non lance (>6h sur GPU 24GB).")
        return None
    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_pre.py ignore (poids charges depuis data/ckpt/pre/).")
        return True
    # Article: per_device_batch=16, grad_accum=4 -> effective batch 64.
    batch, grad_accum = scaled_batch(cfg, per_device_batch=16, grad_accum=4)
    cmd = accelerate_launch_cmd(cfg, "train_pre.py") + [
        "--dataset", "redial",
        "--num_train_epochs", "10",
        "--gradient_accumulation_steps", str(grad_accum),
        "--per_device_train_batch_size", str(batch),
        "--per_device_eval_batch_size", "64",
        "--num_warmup_steps", "1389",
        "--max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--learning_rate", "5e-4",
        "--seed", str(cfg["seed"]),
        "--num_workers", str(cfg.get("num_workers", 16)),
        "--nei_mer",
    ]
    if cfg.get("enable_torch_compile", False):
        cmd.append("--enable_torch_compile")
    return _run(cmd, cwd=cfg["src_emo_dir"], description="train_pre")


## 5. Sous-tache Emotion-aware Item Recommendation (Section 4.2)

Cette etape entraine le module de recommandation avec :

- representations locales / globales emotion-aware (Eq. 3-6) ;
- **feedback-aware item reweighting** (Eq. 7) avec les poids `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- option `--use_sentiment` pour utiliser la classification fine-tunee GPT-2 du README.

In [10]:
# Cellule 8 - Emotion-aware Item Recommendation (train_rec.py)
def _resolve_rec_prompt_encoder(src_emo, preference=("best", "final", "")):
    """Retourne le premier dossier `data/saved/rec/<sub>` present (ex: best -> final -> racine).

    Args:
        preference: ordre de recherche. "" = data/saved/rec (racine, ancien format).
    """
    root = src_emo / "data" / "saved" / "rec"
    for sub in preference:
        p = root / sub if sub else root
        if p.exists():
            return p
    return None


def run_recommendation_training(cfg):
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_rec.py non lance (entrainement GPU necessaire).")
        return None

    src_emo = cfg["src_emo_dir"]
    prompt_encoder_candidates = [
        src_emo / "data" / "saved" / "pre" / "final" / "model.pt",
        src_emo / "data" / "saved" / "pre" / "model.pt",
    ]
    pre_ckpt = next((p for p in prompt_encoder_candidates if p.exists()), None)
    if pre_ckpt is None:
        print("[skip] train_rec: checkpoint pre absent (attendus: data/saved/pre/final/model.pt ou data/saved/pre/model.pt).")
        return False
    pre_prompt_encoder_dir = str(pre_ckpt.parent.relative_to(src_emo))

    rec_compile = cfg.get("rec_enable_torch_compile")
    if rec_compile is None:
        rec_compile = cfg.get("enable_torch_compile", False)
    rec_mp = cfg.get("rec_mixed_precision", cfg.get("mixed_precision", "no"))
    print(f"[rec] mixed_precision={rec_mp} enable_torch_compile={rec_compile} rec_batch_scale={cfg.get('rec_batch_scale', cfg.get('batch_scale', 1.0))}")

    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_rec.py training ignore (poids charges depuis data/ckpt/rec/).")
        train_ok = True
    else:
        batch, grad_accum = scaled_batch_phase(cfg, per_device_batch=cfg["batch_rec"] // 8, grad_accum=8, phase="rec")
        train_cmd = accelerate_launch_cmd_phase(cfg, "train_rec.py", phase="rec") + [
            "--dataset", "redial_gen",
            "--n_prefix_rec", "10",
            "--num_train_epochs", "5",
            "--per_device_train_batch_size", str(batch),
            "--per_device_eval_batch_size", "32",
            "--gradient_accumulation_steps", str(grad_accum),
            "--num_warmup_steps", "530",
            "--context_max_length", "200",
            "--prompt_max_length", "200",
            "--entity_max_length", "32",
            "--prompt_encoder", pre_prompt_encoder_dir,
            "--learning_rate", str(cfg["lr_dialogpt"]),
            "--seed", "8",
            "--num_workers", str(cfg.get("num_workers", 16)),
            "--like_score", str(cfg["feedback_weights"]["like"]),
            "--dislike_score", str(cfg["feedback_weights"]["dislike"]),
            "--notsay_score", str(cfg["feedback_weights"]["not say"]),
            "--weighted_loss",
            "--nei_mer",
            "--use_sentiment",
        ]
        if rec_compile:
            train_cmd.append("--enable_torch_compile")
        train_ok = _run(train_cmd, cwd=src_emo, description="train_rec")

    if train_ok is False:
        return False

    # Export obligatoire pour merge_rec.py: genere save/redial_rec/rec.json via train_rec --test.
    # Recommandation native article : on privilegie `rec/best` (selection par valid loss),
    # avec fallback sur `rec/final` puis `rec/` racine.
    pref = cfg.get("rec_export_checkpoint", "best")
    order = [pref, "final", "best", ""] if pref in ("best", "final") else ["best", "final", ""]
    # dedoublonner en gardant l'ordre
    seen = set(); ordered = []
    for s in order:
        if s not in seen:
            seen.add(s); ordered.append(s)
    rec_prompt_dir = _resolve_rec_prompt_encoder(src_emo, preference=tuple(ordered))
    if rec_prompt_dir is None:
        print("[skip] train_rec export: prompt_encoder rec absent (attendus: data/saved/rec/{best,final,}/).")
        return False
    chosen_label = rec_prompt_dir.name if rec_prompt_dir.name in ("best", "final") else "root"
    print(f"[rec] export checkpoint -> {rec_prompt_dir} (label={chosen_label})")

    export_cmd = accelerate_launch_cmd_phase(cfg, "train_rec.py", phase="rec") + [
        "--dataset", "redial_gen",
        "--n_prefix_rec", "10",
        "--test",
        "--num_train_epochs", "1",
        "--max_train_steps", "1",
        "--num_warmup_steps", "0",
        "--per_device_train_batch_size", "1",
        "--gradient_accumulation_steps", "1",
        "--learning_rate", str(cfg["lr_dialogpt"]),
        "--per_device_eval_batch_size", "32",
        "--context_max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--prompt_encoder", str(rec_prompt_dir.relative_to(src_emo)),
        "--seed", "8",
        "--num_workers", str(cfg.get("num_workers", 16)),
        "--like_score", str(cfg["feedback_weights"]["like"]),
        "--dislike_score", str(cfg["feedback_weights"]["dislike"]),
        "--notsay_score", str(cfg["feedback_weights"]["not say"]),
        "--weighted_loss",
        "--nei_mer",
        "--use_sentiment",
    ]
    if rec_compile:
        export_cmd.append("--enable_torch_compile")
    return _run(export_cmd, cwd=src_emo, description="train_rec_export_rec_json")


## 6. Sous-tache Emotion-aligned Response Generation (Section 4.3)

Deux variantes sont fournies dans le depot ECR :

- **ECR[DialoGPT]** : `train_emp.py` puis `infer_emp.py`. On injecte le prompt emotion-aligned (Eq. 8) construit a partir des reviews IMDb filtrees (34,953 reviews / 4,092 films).
- **ECR[Llama 2-Chat]** : fine-tuning LoRA via LLaMA Board sur `src_emo/data/emo_data/llama_train.json` + `llama_test.json` (2,459 reviews / 1,553 films).

La fonction ci-dessous pilote la variante DialoGPT (la variante Llama est fine-tunee via un outil externe).

In [11]:
# Cellule 9 - Emotion-aligned Response Generation (DialoGPT)
def run_response_generation_training(cfg):
    if cfg["dry_run"]:
        print("DRY_RUN=True -> pipeline generation non lance.")
        return None
    src_emo = cfg["src_emo_dir"]
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"

    # merge_rec.py lit ../../save/redial_rec/rec.json -> cwd doit etre redial_gen.
    rec_json = src_emo / "save" / "redial_rec" / "rec.json"
    if (not rec_json.exists()) or rec_json.stat().st_size == 0:
        print(f"[skip] merge_rec: fichier absent/vide ({rec_json}). train_rec n'a pas produit de recommandations.")
        return False

    if _run(["python", "merge_rec.py"], cwd=redial_gen, description="merge_rec") is False:
        return False
    if _run(["python", "imdb_review_entity_filter.py"], cwd=src_emo, description="imdb_review_entity_filter") is False:
        return False
    if _run(["python", "process_empthetic.py"], cwd=redial, description="process_empthetic") is False:
        return False

    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_emp.py ignore, on passe direct a l'inference.")
    else:
        # Article: per_device_batch=20, grad_accum=1 -> effective batch 20.
        # NOTE: on NE scale PAS l'emp (grad_accum=1 => pas possible de preserver l'effective batch),
        # pour rester coherent avec l'article. batch_scale ne s'applique qu'a pre/rec.
        batch_train = 20
        grad_accum = 1
        # eval/generate est tres gourmand avec torch.compile + cudagraphs (OOM 32GB a batch=64).
        # On reduit a une valeur safe; eval batch ne change pas les metriques.
        batch_eval = 16
        train_cmd = accelerate_launch_cmd(cfg, "train_emp.py") + [
            "--dataset", "redial",
            "--num_train_epochs", "15",
            "--gradient_accumulation_steps", str(grad_accum),
            "--ignore_pad_token_for_loss",
            "--per_device_train_batch_size", str(batch_train),
            "--per_device_eval_batch_size", str(batch_eval),
            "--num_warmup_steps", "9965",
            "--context_max_length", "150",
            "--resp_max_length", "150",
            "--learning_rate", str(cfg["lr_dialogpt"]),
            "--num_workers", str(cfg.get("num_workers", 16)),
        ]
        if cfg.get("enable_torch_compile", False):
            train_cmd.append("--enable_torch_compile")
        if _run(train_cmd, cwd=src_emo, description="train_emp") is False:
            return False

    # Garde: infer_emp a besoin du checkpoint train_emp (save/emp/model.safetensors).
    emp_dir = src_emo / "data" / "saved" / "emp"
    emp_ckpt = next((emp_dir / n for n in ("model.safetensors", "pytorch_model.bin") if (emp_dir / n).exists()), None)
    if emp_ckpt is None:
        print(f"[skip] infer_emp: checkpoint train_emp absent ({emp_dir}).")
        return False

    infer_cmd = accelerate_launch_cmd(cfg, "infer_emp.py") + [
        "--dataset", "redial_gen",
        "--split", "test",
        "--per_device_eval_batch_size", "64",
        "--context_max_length", "150",
        "--resp_max_length", "150",
        "--num_workers", str(cfg.get("num_workers", 16)),
    ]
    return _run(infer_cmd, cwd=src_emo, description="infer_emp")


## 6bis. Table 2 -- Llama 2-7B-Chat, ECR[Llama 2-Chat] LoRA et scorer local Qwen

L'article evalue 6 systemes (Table 2 + Appendix E) :

| Systeme | Dans ce notebook |
|---|---|
| `UniCRS` generation | fallback article (pas de ckpt vanille fourni) |
| `GPT-3.5-turbo-instruct` | fallback article (API payante) |
| `GPT-3.5-turbo` | fallback article (API payante) |
| `Llama 2-7B-Chat` zero-shot | **reproduit** (cellule ci-dessous) |
| `ECR[DialoGPT]` | **reproduit** (via `infer_emp.py`) |
| `ECR[Llama 2-Chat]` | **reproduit** via LoRA fine-tune (`src/ecr/llama_lora.py`) |

Le scorer GPT-4-turbo de l'Appendix E est remplace par **Qwen2.5-32B-Instruct-AWQ** servi via vLLM (gratuit, local, 4-bit AWQ ~20 GB sur 32 GB). On reporte un **Cohen kappa au niveau classement** entre ce scorer local et les annotateurs humains de l'article, pour documenter la validite du proxy.

In [12]:
# Cellule 9b - Llama 2-7B-Chat zero-shot (baseline Table 2)
#
# Charge `test_data_dbpedia_emo.jsonl` (ou l'equivalent produit par prepare_redial),
# applique le template Llama 2 Chat (<s>[INST]<<SYS>>...[/INST]) et ecrit
# `results/generations/llama2_zero_shot.jsonl` (format attendu par le scorer).
#
# Deux backends :
#   - "hf" (defaut) : transformers.pipeline bf16 + Flash-Attn 2. Simple, fiable.
#     ~3-5 samples/s sur 5090 pour Llama-2-7B-Chat.
#   - "vllm" : serveur vLLM local. ~30-50 samples/s (10x). Besoin de ~15 GB VRAM
#     libre. Utile si > 500 samples. Le serveur est demarre par launch_llama_vllm.

_LLAMA_VLLM_PROC = [None]  # handle du process, pour teardown


def _is_port_open(port, host="localhost", timeout=1.0):
    """True si un serveur ecoute deja sur (host, port) (reutilisation possible)."""
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(timeout)
    try:
        return s.connect_ex((host, int(port))) == 0
    finally:
        try:
            s.close()
        except Exception:
            pass


def launch_llama_vllm(cfg):
    """Lance `vllm serve <cfg['llama_base_model']>` en background.

    Par defaut `cfg['llama_base_model']` pointe sur le miroir public
    `NousResearch/Llama-2-7b-chat-hf` (poids identiques bit-a-bit au repo gated
    `meta-llama/Llama-2-7b-chat-hf`).

    Retourne True si le serveur est pret, False sinon. Reutilise un serveur
    deja en ecoute sur `cfg['llama_serve_port']` si present (utile quand un
    `vllm serve` a ete lance manuellement dans un terminal).

    Mutation du cfg : on NE change PAS `llama_backend` ici (responsabilite de
    l'appelant) pour garder la fonction pure.
    """
    if not cfg.get("llama_vllm_autostart"):
        return False
    port = cfg["llama_serve_port"]
    if _LLAMA_VLLM_PROC[0] is not None and _LLAMA_VLLM_PROC[0].poll() is None:
        print("[llama-vllm] deja en cours (process notebook)")
        return True
    if _is_port_open(port):
        print(f"[llama-vllm] port {port} deja occupe -> reutilisation du serveur externe")
        return True
    from src.ecr import scorer as sc
    # gpu_memory_utilization : a partir de vLLM 0.19, le profiler memoire tient
    # compte des CUDA graphs (avant il les ignorait). Llama-2-7B bf16 = 12.55 GB
    # de poids + ~12 GB de CUDA graphs : a 0.45 * 32 GB = 14.4 GB, le budget KV
    # cache devient NEGATIF (ValueError au demarrage). vLLM recommande lui-meme
    # 0.85 dans son log d'erreur. On garde ce defaut ; le scorer Qwen (run
    # sequentiellement APRES teardown Llama) prendra la VRAM a son tour.
    log_path = cfg["logs_dir"] / "vllm_llama.log"
    proc = sc.launch_vllm_server(
        model=cfg["llama_base_model"],
        port=port,
        quantization=None,  # Llama 2-7B-Chat en bf16 (pas de quant)
        max_model_len=2048,
        gpu_memory_utilization=float(cfg.get("llama_vllm_gmu", 0.85)),
        log_path=log_path,
        extra_args=["--dtype", "bfloat16", "--enable-prefix-caching"],
    )
    if proc is None:
        print("[llama-vllm] echec demarrage (binaire vllm introuvable ?) -> fallback backend hf")
        return False
    _LLAMA_VLLM_PROC[0] = proc
    timeout = int(cfg.get("llama_vllm_ready_timeout", 1800))
    print(f"[llama-vllm] wait_vllm_ready timeout={timeout}s "
          f"(cold cache : ~14 min download + 1-2 min warmup ; cache chaud : ~10 s)")

    # Poll manuel : on surveille a la fois (a) le port HTTP ET (b) la sante du
    # process vllm. Si `proc.poll()` retourne un code non-None, inutile
    # d'attendre 30 min : le process est deja mort (ex: ValueError KV cache,
    # OOM CUDA, fichier modele corrompu...). On bascule immediatement en HF.
    import time as _time
    import urllib.request as _urlreq
    import urllib.error as _urlerr
    deadline = _time.time() + timeout
    ready_url = f"http://localhost:{port}/health"
    while _time.time() < deadline:
        exit_code = proc.poll()
        if exit_code is not None:
            # Process mort -> on extrait les derniers 2 KB de log pour expliquer.
            try:
                tail = log_path.read_text(errors="replace")[-2048:]
            except Exception:
                tail = "(log illisible)"
            print(f"[llama-vllm] vllm a crashe (exit={exit_code}) -> fallback backend hf")
            print(f"[llama-vllm] --- tail {log_path} ---\n{tail}\n--- end ---")
            _LLAMA_VLLM_PROC[0] = None
            return False
        try:
            with _urlreq.urlopen(ready_url, timeout=2) as r:
                if r.status == 200:
                    print(f"[llama-vllm] ready on :{port}")
                    return True
        except (_urlerr.URLError, OSError):
            pass
        _time.sleep(5)

    # Timeout depasse sans crash ni ready -> on tue tout le process group
    # (vLLM + enfants EngineCore/NCCL) et on bascule.
    print(f"[llama-vllm] timeout {timeout}s -> fallback backend hf (voir {log_path})")
    from src.ecr.scorer import kill_vllm_proc_tree
    kill_vllm_proc_tree(proc, timeout=10)
    _LLAMA_VLLM_PROC[0] = None
    return False


def teardown_llama_vllm():
    """A appeler apres run_llama_lora_generate pour liberer la VRAM avant Qwen.

    Kill le process group entier (vLLM + enfants EngineCore) via `killpg`. Sans
    ca, un simple `proc.terminate()` laissait des `[python3] <defunct> ppid=1`
    dans le conteneur (enfants orphelins re-adoptes par init).
    """
    if _LLAMA_VLLM_PROC[0] is None:
        return
    from src.ecr.scorer import kill_vllm_proc_tree
    print("[llama-vllm] terminate (process group)")
    kill_vllm_proc_tree(_LLAMA_VLLM_PROC[0], timeout=30)
    _LLAMA_VLLM_PROC[0] = None

def run_llama_zero_shot(cfg, limit=1000):
    """Genere les reponses Llama 2-7B-Chat zero-shot pour Table 2."""
    if cfg["dry_run"] or not cfg.get("run_llama_zero_shot"):
        print("run_llama_zero_shot desactive -> ligne Llama 2-7B-Chat en fallback article.")
        return None
    from src.ecr import llama_runner as lr

    gen_dir = Path(cfg["generations_dir"])
    gen_dir.mkdir(parents=True, exist_ok=True)
    out_path = gen_dir / "llama2_zero_shot.jsonl"
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[llama-zs] deja present: {out_path}")
        return out_path

    redial_gen = cfg["src_emo_dir"] / "data" / "redial_gen"
    candidates = [
        redial_gen / "test_data_dbpedia_emo.jsonl",
        redial_gen / "test_data_dbpedia.jsonl",
        redial_gen / "test_data.jsonl",
    ]
    jsonl = next((p for p in candidates if p.exists()), None)
    if jsonl is None:
        print(f"[llama-zs] test set introuvable parmi {[str(p) for p in candidates]}")
        return None

    samples = lr.load_test_samples(jsonl, limit=limit)
    print(f"[llama-zs] {len(samples)} exemples -> {out_path}")

    # --- Politique VRAM : vLLM OBLIGATOIRE pour le zero-shot ---
    # Rationale : charger Llama-2-7B bf16 (12.5 GB) dans le notebook kernel
    # rend impossible le lancement ulterieur de Qwen 32B AWQ (20 GB) car la
    # CUDA context du kernel Python reste sur le GPU (~0.5-2 GB residuels)
    # jusqu'a shutdown kernel -- meme apres gc.collect + empty_cache. En
    # imposant vLLM (subprocess dedie, process-group isole, kill propre via
    # killpg), on garantit que 100% de la VRAM est rendue au driver apres la
    # phase Llama, et Qwen peut demarrer sans conflit.
    backend = cfg.get("llama_backend", "vllm")
    if backend != "vllm":
        print(f"[llama-zs] backend={backend!r} force a 'vllm' (contrainte VRAM, voir cellule).")
        backend = "vllm"
    ok = launch_llama_vllm(cfg)
    if not ok:
        print("[llama-zs] vLLM n'a pas pu demarrer. On ARRETE la phase Llama zero-shot")
        print("[llama-zs] plutot que de charger Llama inline (fuite VRAM ulterieure).")
        print("[llama-zs] Verifier logs/vllm_llama.log, la VRAM disponible, et relancer.")
        return None

    outputs = lr.generate_vllm(
        samples,
        base_url=f"http://localhost:{cfg['llama_serve_port']}/v1",
        model_name=cfg["llama_base_model"],
    )

    # Enrichit chaque sortie avec l'history textuel (utile pour le scorer).
    for out, sample in zip(outputs, samples):
        out["history"] = sample.history
    lr.dump_generations(outputs, out_path)
    print(f"[llama-zs] OK -> {out_path}")
    # NB : pas de gc/empty_cache ici -- aucun tenseur n'est dans ce process
    # (vLLM est un subprocess). La VRAM sera rendue au teardown_llama_vllm().
    return out_path

In [13]:
# Cellule 9c - ECR[Llama 2-Chat] : LoRA fine-tune + generation
#
# Corpus : `emo_data/llama_train.json` (2,459 reviews empathiques filtrees par
# les auteurs, Section 4.3 + 5.1) avec le prompt "emotion-aligned" (Eq. 8).
#
# Hyperparametres fideles a la Section 5.4 :
#   - LoRA r=16, alpha=32, dropout=0.05, modules q/k/v/o_proj ;
#   - 15 epochs, lr=5e-5, per_device_batch=4, grad_accum=4 -> eff batch 16 ;
#   - bf16 sur RTX 5090 (32 GB) ; 4-bit QLoRA dispo en fallback (cfg['lora_use_4bit']).
#
# Duree estimee : 2-3h sur 5090 bf16 (corpus tres petit).

def run_llama_lora_train(cfg):
    """Fine-tune le LoRA ECR[Llama 2-Chat] sur llama_train.json."""
    if cfg["dry_run"] or not cfg.get("run_llama_lora"):
        print("run_llama_lora desactive -> ligne ECR[Llama 2-Chat] en fallback article.")
        return None

    lora_dir = Path(cfg["llama_lora_dir"])
    adapter_file = lora_dir / "adapter_model.safetensors"
    if adapter_file.exists() or (lora_dir / "adapter_model.bin").exists():
        print(f"[lora] adapter deja entraine: {lora_dir}")
        return lora_dir

    train_json = cfg["emo_data_dir"] / "llama_train.json"
    if not train_json.exists():
        alt = cfg["src_emo_dir"] / "data" / "emo_data" / "llama_train.json"
        if alt.exists():
            train_json = alt
    if not train_json.exists():
        print(f"[lora] llama_train.json absent (cherche dans {train_json}) -> skip")
        return None

    from src.ecr.llama_lora import train_lora
    import os as _os

    hf_token = _os.environ.get("HF_TOKEN")
    base = cfg.get("llama_local_dir")
    base = str(base) if base and Path(base).exists() else cfg["llama_base_model"]

    print(f"[lora] base={base} train={train_json} -> {lora_dir}")
    result = train_lora(
        base_model=base,
        train_json=train_json,
        output_dir=lora_dir,
        lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["lora_target_modules"],
        epochs=cfg["lora_epochs"],
        lr=cfg["lr_llama"],
        per_device_batch=cfg["lora_per_device_batch"],
        grad_accum=cfg["lora_grad_accum"],
        hf_token=hf_token,
        use_4bit=cfg.get("lora_use_4bit", False),
        use_flash_attn_2=cfg.get("use_flash_attn_2", True),
    )
    # Liberation VRAM : le modele base + LoRA reste dans `train_lora` closure.
    # On force un GC + empty_cache avant de recharger pour la generation.
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
    except Exception:
        pass
    return result


def run_llama_lora_generate(cfg, limit=1000):
    """Genere les reponses ECR[Llama 2-Chat] sur le test set."""
    if cfg["dry_run"] or not cfg.get("run_llama_lora"):
        return None

    gen_dir = Path(cfg["generations_dir"])
    gen_dir.mkdir(parents=True, exist_ok=True)
    out_path = gen_dir / "ecr_llama_chat.jsonl"
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[lora-gen] deja present: {out_path}")
        return out_path

    lora_dir = Path(cfg["llama_lora_dir"])
    if not (lora_dir / "adapter_config.json").exists():
        print(f"[lora-gen] pas d'adapter dans {lora_dir} -> skip")
        return None

    redial_gen = cfg["src_emo_dir"] / "data" / "redial_gen"
    candidates = [
        redial_gen / "test_data_dbpedia_emo.jsonl",
        redial_gen / "test_data_dbpedia.jsonl",
    ]
    jsonl = next((p for p in candidates if p.exists()), None)
    if jsonl is None:
        print(f"[lora-gen] test set introuvable -> skip")
        return None

    from src.ecr import llama_runner as lr

    base = cfg.get("llama_local_dir")
    base = str(base) if base and Path(base).exists() else cfg["llama_base_model"]
    samples = lr.load_test_samples(jsonl, limit=limit)
    outputs = lr.generate_hf(
        samples,
        model_dir=base,
        lora_dir=str(lora_dir),
        use_flash_attn_2=cfg.get("use_flash_attn_2", True),
    )
    for out, sample in zip(outputs, samples):
        out["history"] = sample.history
    lr.dump_generations(outputs, out_path)
    print(f"[lora-gen] OK -> {out_path}")

    # Liberation VRAM avant la phase 4 (scorer Qwen 32B AWQ qui prend ~20 GB).
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
    except Exception:
        pass
    return out_path

In [14]:
# Cellule 9d - Scorer Qwen2.5-32B-AWQ local (remplace GPT-4-turbo Appendix E)
#
# IMPORTANT : le serveur vLLM monopolise ~20 GB VRAM. Il DOIT etre lance APRES
# les phases d'entrainement et d'inference Llama (eviter tout conflit VRAM).
# On fait donc un `torch.cuda.empty_cache()` explicite avant, et on tue le
# process vLLM en fin de cellule.
#
# 4 systemes x 200 exemples x 5 rubriques = 4000 notes = ~2h15 sur 5090.

def run_llm_scorer(cfg):
    """Note chaque systeme avec Qwen2.5 et produit `subjective_metrics_llm.csv`."""
    if cfg["dry_run"] or not cfg.get("run_llm_scorer"):
        print("run_llm_scorer desactive -> subjective_metrics_llm.csv en fallback article.")
        return None

    gen_dir = Path(cfg["generations_dir"])
    systems = {
        "ECR[DialoGPT]":     gen_dir / "ecr_dialogpt.jsonl",
        "Llama 2-7B-Chat":   gen_dir / "llama2_zero_shot.jsonl",
        "ECR[Llama 2-Chat]": gen_dir / "ecr_llama_chat.jsonl",
    }
    available = {k: v for k, v in systems.items() if v.exists()}
    if not available:
        print("[scorer] aucune generation disponible -> fallback article")
        return None

    # Liberation agressive de la VRAM avant de lancer Qwen 32B AWQ (~20 GB).
    # On accepte de perdre le cache CUDA ; pire cas on le rebuild cote vLLM.
    min_free_gb = float(cfg.get("scorer_min_free_gb", 22.0))
    free_gb = None
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
            free, total = _torch.cuda.mem_get_info()
            free_gb = free / 1e9
            total_gb = total / 1e9
            print(f"[scorer] VRAM libre avant vLLM : {free_gb:.1f}/{total_gb:.1f} GB "
                  f"(seuil={min_free_gb:.1f} GB)")
    except Exception as exc:
        print(f"[scorer] check VRAM echoue (non bloquant): {exc}")

    # Abort bloquant (pas warning) : sans cette garde, on voyait Qwen AWQ
    # demarrer puis OOM apres 5-10 min de download + compile, perdre tout le
    # travail de la session. Mieux vaut echouer en 0s avec un message clair.
    # Override possible via `cfg["scorer_min_free_gb"] = 18.0` si tu veux
    # tenter le coup (OOM probable, mais parfois ca passe selon fragmentation).
    if free_gb is not None and free_gb < min_free_gb:
        print(f"[scorer] ABORT : VRAM libre {free_gb:.1f} GB < seuil {min_free_gb:.1f} GB.")
        print("[scorer] Causes probables :")
        print("         - Un modele HF charge dans le kernel notebook (restart kernel ?).")
        print("         - Un vllm Llama pas completement termine (teardown_llama_vllm).")
        print("         - Autre process (nvidia-smi dans un terminal : ps -ef | grep python).")
        print(f"[scorer] Pour forcer malgre tout : cfg['scorer_min_free_gb'] = {free_gb:.1f}")
        return None

    from src.ecr import scorer as sc
    import json as _json
    import time as _time
    import atexit

    proc = sc.launch_vllm_server(
        model=cfg["scorer_model"],
        port=cfg["scorer_port"],
        quantization="awq",
        max_model_len=cfg["scorer_max_model_len"],
        gpu_memory_utilization=0.9,
        log_path=cfg["logs_dir"] / "vllm_scorer.log",
    )
    if proc is None:
        print("[scorer] serveur vLLM n'a pas pu demarrer -> fallback article")
        return None
    # atexit fallback : si le notebook crash, on kill quand meme le process tree.
    atexit.register(lambda: sc.kill_vllm_proc_tree(proc, timeout=10))
    try:
        if not sc.wait_vllm_ready(port=cfg["scorer_port"], timeout=900):
            print("[scorer] serveur vLLM non pret apres 15min -> abandon")
            return None

        from openai import OpenAI
        client = OpenAI(base_url=f"http://localhost:{cfg['scorer_port']}/v1", api_key="EMPTY")

        per_system = {}
        for sys_name, path in available.items():
            rows = []
            with path.open() as f:
                for line in f:
                    rows.append(_json.loads(line))
            print(f"[scorer] {sys_name}: {len(rows)} exemples dispo, sample={cfg['scorer_sample_size']}")
            t0 = _time.time()
            df = sc.score_system(
                client,
                model=cfg["scorer_model"],
                generations=rows,
                sample_size=cfg["scorer_sample_size"],
                seed=cfg["seed"],
                max_workers=cfg.get("scorer_concurrency", 16),
            )
            dt = _time.time() - t0
            print(f"[scorer] {sys_name}: {len(df)} notes en {dt/60:.1f}min")
            per_system[sys_name] = df
            df.to_csv(cfg["results_dir"] / f"scorer_raw_{sys_name.replace(' ', '_').replace('[','').replace(']','')}.csv", index=False)

        df_subj = sc.aggregate_subjective(per_system)
        df_subj.to_csv(cfg["results_dir"] / cfg["subjective_llm_file"], index=False)
        print(f"[scorer] -> {cfg['results_dir'] / cfg['subjective_llm_file']}")
        print(df_subj)
        return df_subj
    finally:
        # kill_vllm_proc_tree : tue aussi les enfants (EngineCore/NCCL) via
        # killpg(pgid, SIGTERM) puis SIGKILL apres timeout.
        sc.kill_vllm_proc_tree(proc, timeout=30)

In [15]:
# Cellule 9e - Cohen kappa scorer local vs annotateurs humains (Section 5.6)
#
# On compare le CLASSEMENT (rank) des systemes par metrique, pas les valeurs
# absolues (echelles differentes : LLM 0-9 strict, humains 0-9 mais biais
# differents). Kappa > 0.6 = accord substantiel (chiffre reference article).

def compute_scorer_kappa(cfg, df_subj_llm, df_subj_human):
    if cfg["dry_run"] or not cfg.get("run_kappa"):
        return None
    try:
        from sklearn.metrics import cohen_kappa_score
    except ImportError:
        print("[kappa] scikit-learn indisponible -> skip")
        return None

    metrics = ["Emo Int", "Emo Pers", "Log Pers", "Info", "Life"]
    rows = []
    for m in metrics:
        if m not in df_subj_llm.columns or m not in df_subj_human.columns:
            continue
        llm = df_subj_llm.set_index("Model")[m]
        human = df_subj_human.set_index("Model")[m]
        common = llm.index.intersection(human.index)
        if len(common) < 3:
            continue
        rk_llm = llm[common].rank(method="min").astype(int)
        rk_human = human[common].rank(method="min").astype(int)
        kappa = cohen_kappa_score(rk_llm.tolist(), rk_human.tolist())
        rows.append({"metric": m, "n_models": len(common), "kappa_rank": kappa})
    df_kappa = pd.DataFrame(rows)
    if not df_kappa.empty:
        df_kappa.to_csv(cfg["results_dir"] / "scorer_kappa.csv", index=False)
        print("=== Cohen kappa scorer local (Qwen2.5-32B-AWQ) vs humains article ===")
        print(df_kappa)
        mean_k = df_kappa["kappa_rank"].mean()
        level = "SUBSTANTIEL" if mean_k > 0.6 else ("MODERE" if mean_k > 0.4 else "FAIBLE")
        print(f"kappa moyen = {mean_k:.3f} ({level})")
    return df_kappa

In [16]:
# Cellule 9f - Parse des logs train_rec -> objective_metrics.csv + ablation_metrics.csv
#
# A lancer APRES run_recommendation_sweep. Extrait les 7 metriques Table 1 de
# chaque log et complete avec les valeurs article pour les baselines non
# reproduites (KBRD, KGSF, RevCore, UCCR).
#
# Cette fonction est robuste a l'absence de logs : si aucun run du sweep n'a
# produit de log parsable, elle retourne None et load_results_data utilisera
# le fallback article. AUCUNE exception n'est propagee pour ne pas casser le
# pipeline (on accepte une Table 1 incomplete).

def build_results_from_logs(cfg):
    if cfg["dry_run"]:
        return None
    try:
        from src.ecr.metrics_parser import write_rec_results
    except ImportError as exc:
        print(f"[parse] import metrics_parser echoue: {exc}")
        return None

    logs_dir = Path(cfg["logs_dir"])
    if not logs_dir.exists():
        print(f"[parse] {logs_dir} absent -> aucun log a parser")
        return None

    try:
        run_names = [v["name"] for v in REC_VARIANTS]
    except NameError:
        print("[parse] REC_VARIANTS non defini (notebook flo-stuff, single run) -> skip")
        return None
    available_logs = [n for n in run_names if (logs_dir / f"train_rec_{n}.log").exists()]
    if not available_logs:
        print(f"[parse] aucun log train_rec_*.log dans {logs_dir} -> fallback article")
        return None

    # _fallback_objective est defini dans code-17-load (cellule suivante) ;
    # on gere le cas ou il n'est pas encore charge (execution hors-ordre).
    try:
        fallback = _fallback_objective()
    except NameError:
        fallback = None

    try:
        df_obj = write_rec_results(
            logs_dir=logs_dir,
            results_dir=cfg["results_dir"],
            run_names=run_names,
            fallback_objective=fallback,
        )
    except Exception as exc:
        print(f"[parse] echec write_rec_results: {exc}")
        return None

    if df_obj is not None and not df_obj.empty:
        print("=== Table 1 (objective_metrics.csv) ===")
        print(df_obj)
    return df_obj

## 7. Chargement des resultats (Tables 1-3)

`load_results_data` lit des CSV facultatifs dans `results/`. En l'absence de fichiers (cas le plus courant : `DRY_RUN=True`), on retombe sur les valeurs publiees par l'article afin que le notebook reste reproductible :

- Table 1 — recommandation objective (AUC, RT@n, R@n) ;
- Table 2 — generation subjective (Emo Int, Emo Pers, Log Pers, Info, Life) ;
- Table 3 — etude d'ablation (UniCRS, ECR[L], ECR[LS], ECR[LG], ECR).

In [17]:
# Cellule 10 - Chargement des metriques (repli sur les tables de l'article)
def _fallback_objective():
    return pd.DataFrame([
        {"Model": "KBRD",    "AUC": 0.503, "RT@1": 0.040, "RT@10": 0.182, "RT@50": 0.381, "R@1": 0.037, "R@10": 0.175, "R@50": 0.360},
        {"Model": "KGSF",    "AUC": 0.513, "RT@1": 0.043, "RT@10": 0.195, "RT@50": 0.383, "R@1": 0.040, "R@10": 0.182, "R@50": 0.361},
        {"Model": "RevCore", "AUC": 0.514, "RT@1": 0.054, "RT@10": 0.230, "RT@50": 0.410, "R@1": 0.046, "R@10": 0.209, "R@50": 0.390},
        {"Model": "UCCR",    "AUC": 0.499, "RT@1": 0.038, "RT@10": 0.208, "RT@50": 0.423, "R@1": 0.039, "R@10": 0.198, "R@50": 0.407},
        {"Model": "UniCRS",  "AUC": 0.506, "RT@1": 0.052, "RT@10": 0.229, "RT@50": 0.439, "R@1": 0.047, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR",     "AUC": 0.541, "RT@1": 0.055, "RT@10": 0.238, "RT@50": 0.452, "R@1": 0.049, "R@10": 0.220, "R@50": 0.428},
    ])


def _fallback_subjective_llm():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.400, "Emo Pers": 0.942, "Log Pers": 0.793, "Info": 0.673, "Life": 2.241},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 1.706, "Emo Pers": 3.043, "Log Pers": 3.474, "Info": 2.975, "Life": 4.182},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.215, "Emo Pers": 3.754, "Log Pers": 4.782, "Info": 4.147, "Life": 5.338},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 3.934, "Emo Pers": 6.030, "Log Pers": 5.886, "Info": 5.904, "Life": 7.129},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 4.011, "Emo Pers": 4.878, "Log Pers": 4.736, "Info": 5.094, "Life": 5.906},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 6.826, "Emo Pers": 7.724, "Log Pers": 6.702, "Info": 7.653, "Life": 8.063},
    ])


def _fallback_subjective_human():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.947, "Emo Pers": 0.775, "Log Pers": 1.158, "Info": 0.380, "Life": 1.805},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 2.048, "Emo Pers": 2.555, "Log Pers": 3.265, "Info": 1.822, "Life": 3.648},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.890, "Emo Pers": 3.678, "Log Pers": 5.323, "Info": 3.233, "Life": 5.125},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 4.432, "Emo Pers": 6.152, "Log Pers": 6.393, "Info": 5.713, "Life": 7.463},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 5.097, "Emo Pers": 4.817, "Log Pers": 5.398, "Info": 4.628, "Life": 6.385},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 7.130, "Emo Pers": 7.575, "Log Pers": 7.403, "Info": 7.172, "Life": 8.468},
    ])


def _fallback_ablation():
    return pd.DataFrame([
        {"Model": "UniCRS",  "AUC": 0.506, "RT@10": 0.229, "RT@50": 0.439, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR[L]",  "AUC": 0.535, "RT@10": 0.229, "RT@50": 0.444, "R@10": 0.213, "R@50": 0.420},
        {"Model": "ECR[LS]", "AUC": 0.541, "RT@10": 0.232, "RT@50": 0.442, "R@10": 0.216, "R@50": 0.420},
        {"Model": "ECR[LG]", "AUC": 0.535, "RT@10": 0.232, "RT@50": 0.453, "R@10": 0.216, "R@50": 0.428},
        {"Model": "ECR",     "AUC": 0.541, "RT@10": 0.238, "RT@50": 0.452, "R@10": 0.220, "R@50": 0.428},
    ])


def load_results_data(cfg):
    """Charge les metriques depuis `results/` ou retombe sur les tables de l'article."""
    r = cfg["results_dir"]
    r.mkdir(parents=True, exist_ok=True)
    pairs = [
        (r / cfg["objective_file"], _fallback_objective),
        (r / cfg["subjective_llm_file"], _fallback_subjective_llm),
        (r / cfg["subjective_human_file"], _fallback_subjective_human),
        (r / cfg["ablation_file"], _fallback_ablation),
    ]
    frames = []
    for path, fallback in pairs:
        if path.exists():
            print(f"[load] {path.name} depuis results/")
            frames.append(pd.read_csv(path))
        else:
            print(f"[load] {path.name} absent -> fallback article")
            frames.append(fallback())
    return tuple(frames)

In [18]:
# Cellule 10b - Extraction metriques + elapsed time + accumulation vers results/*.csv
import re as _re_metrics
from datetime import datetime as _dt_metrics
import hashlib as _hashlib
import json as _json_metrics

_CONFIG_KEYS = [
    "seed", "batch_scale", "mixed_precision", "num_workers",
    "enable_torch_compile",
    # Overrides specifiques a la phase rec (si present, differencient les runs)
    "rec_batch_scale", "rec_mixed_precision", "rec_enable_torch_compile",
    "batch_rec", "batch_gen",
    "lr_dialogpt", "lr_llama", "dry_run", "use_pretrained_ckpt",
    "feedback_weights", "emotion_embed_dim", "beta", "n_f", "p_nt", "p_ne",
]


def _config_fingerprint(cfg):
    snapshot = {}
    for k in _CONFIG_KEYS:
        if k in cfg:
            v = cfg[k]
            if isinstance(v, dict):
                v = {kk: vv for kk, vv in v.items()}
            snapshot[k] = v
    canon = _json_metrics.dumps(snapshot, sort_keys=True, default=str)
    h = _hashlib.sha1(canon.encode("utf-8")).hexdigest()[:8]
    return snapshot, h


_TS_RE = _re_metrics.compile(r"^(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d+)")


def _parse_last_config(lines):
    for line in reversed(lines):
        m = _re_metrics.search(r"- (\{.*\})$", line)
        if m and "\'seed\'" in m.group(1):
            try:
                s = _re_metrics.sub(r"np\.float64\(([-\d\.eE+]+)\)", r"\1", m.group(1))
                return eval(s, {"__builtins__": {}}, {})
            except Exception:
                return None
    return None


def _parse_metric_dict(line):
    m = _re_metrics.search(r"\{.*\}", line)
    if not m:
        return None
    try:
        s = _re_metrics.sub(r"np\.float64\(([-\d\.eE+]+)\)", r"\1", m.group(0))
        return eval(s, {"__builtins__": {}}, {})
    except Exception:
        return None


def _classify_log(cfg_dict):
    if cfg_dict is None:
        return None
    out = cfg_dict.get("output_dir")
    if out == "data/saved/pre":
        return "pre"
    if out == "data/saved/rec":
        # train_rec --test => log court servant a exporter rec.json (1 step).
        # training complet => log long (num_train_epochs=5, max_train_steps=None).
        if cfg_dict.get("test") is True:
            return "rec_export"
        return "rec_train"
    if out == "data/saved/emp":
        return "emp"
    if out is None and cfg_dict.get("split") == "test":
        return "infer"
    return None


def _last_test_dict(lines):
    last = None
    for line in lines:
        if "test/" in line:
            d = _parse_metric_dict(line)
            if d and any(k.startswith("test/") for k in d.keys()):
                last = d
    return last


def _log_elapsed_seconds(lines):
    first = last = None
    for line in lines:
        m = _TS_RE.match(line)
        if m:
            ts = _dt_metrics.strptime(m.group(1), "%Y-%m-%d %H:%M:%S.%f")
            if first is None:
                first = ts
            last = ts
    if first and last:
        return (last - first).total_seconds()
    return None


def _fmt_elapsed(seconds):
    if seconds is None:
        return ""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def extract_run_metrics(cfg):
    log_dir = cfg["src_emo_dir"] / "log"
    buckets = {"pre": None, "rec_train": None, "rec_export": None, "emp": None, "infer": None}
    elapsed = {k: None for k in buckets}
    paths = {k: None for k in buckets}
    if not log_dir.exists():
        return {"pre": None, "rec": None, "emp": None, "infer": None}, {"pre": None, "rec": None, "rec_export": None, "emp": None, "infer": None}, {"pre": None, "rec": None, "rec_export": None, "emp": None, "infer": None}
    for path in sorted(log_dir.glob("*.log")):
        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue
        lines = text.splitlines()
        kind = _classify_log(_parse_last_config(lines))
        if kind is None:
            continue
        metrics = _last_test_dict(lines)
        if metrics is None and kind != "rec_train":
            continue
        buckets[kind] = metrics  # None OK for rec_train (pas de test bloc)
        elapsed[kind] = _log_elapsed_seconds(lines)
        paths[kind] = path.name
    # Metrics recap: on privilegie l'export rec_test (qui produit le rec.json final) pour "rec".
    # Si absent, on retombe sur rec_train (dernier valid bloc).
    rec_metric = buckets["rec_export"] if buckets["rec_export"] is not None else buckets["rec_train"]
    results = {
        "pre": buckets["pre"],
        "rec": rec_metric,
        "emp": buckets["emp"],
        "infer": buckets["infer"],
    }
    elapsed_out = {
        "pre": elapsed["pre"],
        "rec": elapsed["rec_train"],  # training time reel
        "rec_export": elapsed["rec_export"],
        "emp": elapsed["emp"],
        "infer": elapsed["infer"],
    }
    paths_out = {
        "pre": paths["pre"],
        "rec": paths["rec_train"],
        "rec_export": paths["rec_export"],
        "emp": paths["emp"],
        "infer": paths["infer"],
    }
    return results, elapsed_out, paths_out


def _append_csv(path, row_df):
    import pandas as _pd
    if path.exists():
        old = _pd.read_csv(path)
        new = _pd.concat([old, row_df], ignore_index=True)
    else:
        new = row_df
    new.to_csv(path, index=False)


def export_run_metrics(cfg):
    import pandas as _pd
    r = cfg["results_dir"]
    r.mkdir(parents=True, exist_ok=True)
    metrics, elapsed, paths = extract_run_metrics(cfg)
    run_id = _dt_metrics.now().strftime("%Y-%m-%d_%H-%M-%S")
    config_snapshot, config_hash = _config_fingerprint(cfg)
    config_cols = {
        "config_hash": config_hash,
        "seed": config_snapshot.get("seed"),
        "batch_scale": config_snapshot.get("batch_scale"),
        "mixed_precision": config_snapshot.get("mixed_precision"),
        "num_workers": config_snapshot.get("num_workers"),
        "enable_torch_compile": config_snapshot.get("enable_torch_compile"),
        "lr_dialogpt": config_snapshot.get("lr_dialogpt"),
        "batch_rec": config_snapshot.get("batch_rec"),
        "batch_gen": config_snapshot.get("batch_gen"),
    }
    exported = {}

    total_elapsed = sum(v for v in elapsed.values() if v is not None) or None

    if metrics["rec"] is not None:
        m = metrics["rec"]
        def _v(k):
            v = m.get(k)
            return float(v) if v is not None else float("nan")
        row = {
            "run_id": run_id,
            "Model": "ECR (our run)",
            "AUC": round(_v("test/auc"), 4),
            "RT@1": round(_v("test/recall_true@1"), 4),
            "RT@10": round(_v("test/recall_true@10"), 4),
            "RT@50": round(_v("test/recall_true@50"), 4),
            "R@1": round(_v("test/recall@1"), 4),
            "R@10": round(_v("test/recall@10"), 4),
            "R@50": round(_v("test/recall@50"), 4),
            "elapsed_pre_sec": elapsed["pre"],
            "elapsed_rec_sec": elapsed["rec"],
            "elapsed_emp_sec": elapsed["emp"],
            "elapsed_infer_sec": elapsed["infer"],
            "elapsed_rec_export_sec": elapsed.get("rec_export"),
            "elapsed_total_sec": total_elapsed,
            "elapsed_pre_human": _fmt_elapsed(elapsed["pre"]),
            "elapsed_rec_human": _fmt_elapsed(elapsed["rec"]),
            "elapsed_rec_export_human": _fmt_elapsed(elapsed.get("rec_export")),
            "elapsed_emp_human": _fmt_elapsed(elapsed["emp"]),
            "elapsed_infer_human": _fmt_elapsed(elapsed["infer"]),
            "elapsed_total_human": _fmt_elapsed(total_elapsed),
            "log_pre": paths["pre"],
            "log_rec": paths["rec"],
            "log_rec_export": paths.get("rec_export"),
            "log_emp": paths["emp"],
            "log_infer": paths["infer"],
            **config_cols,
        }
        df = _pd.DataFrame([row])
        out = r / "run_objective_metrics.csv"
        _append_csv(out, df)
        exported["objective"] = out

        article_row = {
            "run_id": "reference",
            "Model": "ECR (article)",
            "AUC": 0.541, "RT@1": 0.055, "RT@10": 0.238, "RT@50": 0.452,
            "R@1": 0.049, "R@10": 0.220, "R@50": 0.428,
            "elapsed_pre_sec": None, "elapsed_rec_sec": None,
            "elapsed_emp_sec": None, "elapsed_infer_sec": None,
            "elapsed_total_sec": None,
            "elapsed_pre_human": "", "elapsed_rec_human": "",
            "elapsed_emp_human": "", "elapsed_infer_human": "",
            "elapsed_total_human": "",
            "log_pre": "paper Table 1", "log_rec": "paper Table 1",
            "log_rec_export": "paper Table 1",
            "log_emp": "paper Table 1", "log_infer": "paper Table 1",
            "elapsed_rec_export_sec": None, "elapsed_rec_export_human": "",
            "config_hash": "paper",
            "seed": None, "batch_scale": None, "mixed_precision": None,
            "num_workers": None, "enable_torch_compile": None,
            "lr_dialogpt": None, "batch_rec": None, "batch_gen": None,
        }
        cmp_path = r / "run_comparison_objective.csv"
        if cmp_path.exists():
            old = _pd.read_csv(cmp_path)
            ours = old[old["Model"] != "ECR (article)"]
            new = _pd.concat([_pd.DataFrame([article_row]), ours, df], ignore_index=True)
        else:
            new = _pd.concat([_pd.DataFrame([article_row]), df], ignore_index=True)
        new.to_csv(cmp_path, index=False)
        exported["comparison_objective"] = cmp_path

    if metrics["infer"] is not None:
        m = metrics["infer"]
        row = {
            "run_id": run_id,
            "Model": "ECR (our run)",
            "bleu@1": float(m.get("test/bleu@1", float("nan"))),
            "bleu@2": float(m.get("test/bleu@2", float("nan"))),
            "bleu@3": float(m.get("test/bleu@3", float("nan"))),
            "bleu@4": float(m.get("test/bleu@4", float("nan"))),
            "dist@1": float(m.get("test/dist@1", float("nan"))),
            "dist@2": float(m.get("test/dist@2", float("nan"))),
            "dist@3": float(m.get("test/dist@3", float("nan"))),
            "dist@4": float(m.get("test/dist@4", float("nan"))),
            "item_ratio": float(m.get("test/item_ratio", float("nan"))),
            "sent_cnt": int(m.get("test/sent_cnt", 0)),
            "elapsed_infer_sec": elapsed["infer"],
            "elapsed_infer_human": _fmt_elapsed(elapsed["infer"]),
            "source_log": paths["infer"],
            **config_cols,
        }
        df = _pd.DataFrame([row])
        out = r / "run_generation_metrics.csv"
        _append_csv(out, df)
        exported["generation"] = out

    if metrics["pre"] is not None:
        m = metrics["pre"]
        row = {
            "run_id": run_id,
            "phase": "pre",
            "source_log": paths["pre"],
            "elapsed_pre_sec": elapsed["pre"],
            "elapsed_pre_human": _fmt_elapsed(elapsed["pre"]),
            **config_cols,
        }
        for k, v in m.items():
            if k.startswith("test/"):
                row[k.replace("test/", "")] = v
        df = _pd.DataFrame([row])
        out = r / "run_pre_metrics.csv"
        _append_csv(out, df)
        exported["pre"] = out

    summary_row = {
        "run_id": run_id,
        "elapsed_pre_sec": elapsed["pre"],
        "elapsed_rec_sec": elapsed["rec"],
        "elapsed_rec_export_sec": elapsed.get("rec_export"),
        "elapsed_emp_sec": elapsed["emp"],
        "elapsed_infer_sec": elapsed["infer"],
        "elapsed_total_sec": total_elapsed,
        "elapsed_pre_human": _fmt_elapsed(elapsed["pre"]),
        "elapsed_rec_human": _fmt_elapsed(elapsed["rec"]),
        "elapsed_rec_export_human": _fmt_elapsed(elapsed.get("rec_export")),
        "elapsed_emp_human": _fmt_elapsed(elapsed["emp"]),
        "elapsed_infer_human": _fmt_elapsed(elapsed["infer"]),
        "elapsed_total_human": _fmt_elapsed(total_elapsed),
        "log_pre": paths["pre"], "log_rec": paths["rec"],
        "log_rec_export": paths.get("rec_export"),
        "log_emp": paths["emp"], "log_infer": paths["infer"],
        **config_cols,
    }
    df = _pd.DataFrame([summary_row])
    out = r / "run_timing.csv"
    _append_csv(out, df)
    exported["timing"] = out


    hist_path = r / "runs_config_history.jsonl"
    hist_entry = {
        "run_id": run_id,
        "config_hash": config_hash,
        "config_snapshot": config_snapshot,
        "elapsed": elapsed,
        "total_elapsed_sec": total_elapsed,
        "log_paths": paths,
    }
    with hist_path.open("a", encoding="utf-8") as _fh:
        _fh.write(_json_metrics.dumps(hist_entry, default=str) + "\n")
    exported["config_history"] = hist_path

    return metrics, elapsed, exported, paths


In [19]:
# Cellule 10c - Diagnostic dataset (verification native: item_ids + #examples par split)
def diagnose_dataset(cfg, max_rows=200000):
    """Verifie la pipeline ECR : taille du pool d'items + nombre d'exemples par split.

    Ne modifie rien ; ecrit un CSV `results/run_dataset_diagnostic.csv`
    qui s'accumule avec les runs precedents (append).
    """
    import json as _json_diag
    import pandas as _pd_diag
    from datetime import datetime as _dt_diag

    src_emo = cfg["src_emo_dir"]
    redial_gen = src_emo / "data" / "redial_gen"
    redial = src_emo / "data" / "redial"
    entity2id_path = redial / "entity2id.json"
    movie_ids_path = redial / "movie_ids.json"

    info = {"run_id": _dt_diag.now().strftime("%Y-%m-%d_%H-%M-%S")}

    # 1) item_ids (= num_items dans le pool reco)
    if movie_ids_path.exists():
        with movie_ids_path.open() as f:
            movie_ids = _json_diag.load(f)
        info["movie_ids_count"] = len(movie_ids) if isinstance(movie_ids, list) else len(movie_ids.keys())
    else:
        info["movie_ids_count"] = None

    # 2) entity2id -> taille du KG (items + entites)
    if entity2id_path.exists():
        with entity2id_path.open() as f:
            e2i = _json_diag.load(f)
        info["entity_count"] = len(e2i)
    else:
        info["entity_count"] = None

    # 3) #examples par split et dataset_*
    def _count_jsonl(p):
        if not p.exists():
            return None
        n = 0
        with p.open() as f:
            for line in f:
                if line.strip():
                    n += 1
                    if n >= max_rows:
                        break
        return n

    splits = ("train", "valid", "test")
    for split in splits:
        for variant, label in [
            (redial_gen / f"{split}_data_dbpedia_emo.jsonl", f"redial_gen_{split}_dbpedia_emo"),
            (redial_gen / f"{split}_data_processed.jsonl",   f"redial_gen_{split}_processed"),
            (redial / f"{split}_data_dbpedia_emo.jsonl",      f"redial_{split}_dbpedia_emo"),
            (redial / f"{split}_data_processed.jsonl",        f"redial_{split}_processed"),
        ]:
            info[label] = _count_jsonl(variant)

    # 4) rec.json (predictions exporees)
    rec_json = src_emo / "save" / "redial_rec" / "rec.json"
    info["rec_json_lines"] = _count_jsonl(rec_json) if rec_json.exists() else None

    # Persist (append)
    out_dir = cfg["results_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / "run_dataset_diagnostic.csv"
    df = _pd_diag.DataFrame([info])
    if out.exists():
        old = _pd_diag.read_csv(out)
        df = _pd_diag.concat([old, df], ignore_index=True)
    df.to_csv(out, index=False)

    print("=== Diagnostic dataset ===")
    for k, v in info.items():
        print(f"  {k}: {v}")
    print(f"-> {out}")
    return info


In [20]:
# Cellule 11 - EDA : distribution de feedback, labels d'emotion, couverture de reviews
def build_dataset_eda(cfg):
    """Produit les dataframes d'EDA alignes avec la Section 5.1 de l'article."""
    df_feedback = pd.DataFrame({
        "feedback": ["like", "dislike", "not say"],
        "count":    [8110,   490,      1400],  # proportions 81.1 / 4.9 / 14.0 %
    })
    # Estimation des parts des 9 labels d'emotion (Section 4.1.1 - negative = 8%)
    df_emotions = pd.DataFrame({
        "emotion": ["like", "curious", "happy", "grateful", "negative",
                      "neutral", "nostalgia", "agreement", "surprise"],
        "share":   [28.0,   14.0,      12.0,    10.0,       8.0,
                     11.0,    5.0,       7.0,        5.0],  # approximation pedagogique
    })
    df_reviews = pd.DataFrame([
        {"backbone": "DialoGPT",         "reviews": 34953, "movies": 4092},
        {"backbone": "Llama 2-7B-Chat",  "reviews": 2459,  "movies": 1553},
    ])
    df_weights = pd.DataFrame([
        {"feedback": k, "weight": v} for k, v in cfg["feedback_weights"].items()
    ])
    return df_feedback, df_emotions, df_reviews, df_weights


def describe_eda(df_feedback, df_emotions, df_reviews):
    print("=== Feedback distribution (Table Section 5.1) ===")
    print(df_feedback)
    print("\n=== Emotion labels (Section 4.1.1) ===")
    print(df_emotions)
    print("\n=== IMDb review coverage (Section 5.1) ===")
    print(df_reviews)

In [21]:
# Cellule 12 - Analyse et table comparative
def evaluate_results(df_obj, df_subj_llm, df_subj_human):
    best_auc = df_obj.loc[df_obj["AUC"].idxmax()]
    best_rt10 = df_obj.loc[df_obj["RT@10"].idxmax()]
    best_life_llm = df_subj_llm.loc[df_subj_llm["Life"].idxmax()]
    best_life_human = df_subj_human.loc[df_subj_human["Life"].idxmax()]
    return pd.DataFrame([
        {"metric": "best_auc",        "model": best_auc["Model"],        "value": best_auc["AUC"]},
        {"metric": "best_rt10",       "model": best_rt10["Model"],       "value": best_rt10["RT@10"]},
        {"metric": "best_life_llm",   "model": best_life_llm["Model"],   "value": best_life_llm["Life"]},
        {"metric": "best_life_human", "model": best_life_human["Model"], "value": best_life_human["Life"]},
    ])


def build_comparison_table(df_obj, df_subj_llm, df_subj_human):
    llm = df_subj_llm.add_suffix("_llm").rename(columns={"Model_llm": "Model"})
    human = df_subj_human.add_suffix("_human").rename(columns={"Model_human": "Model"})
    return df_obj.merge(llm, on="Model", how="left").merge(human, on="Model", how="left")


def simulate_hyperparam_sweep(df_obj):
    """Petit balayage pedagogique sur le poids `like` (Appendix B)."""
    baseline = df_obj.set_index("Model").loc["ECR", "AUC"]
    sweep = pd.DataFrame({
        "like_weight": [1.0, 1.5, 2.0, 2.5, 3.0],
        "AUC":         [baseline - 0.015, baseline - 0.006, baseline, baseline - 0.004, baseline - 0.011],
    })
    return sweep

## 8. Execution manuelle guidee (recommande)

Cette section decoupe la pipeline en etapes pour faciliter les relances partielles.

**Ordre conseille** (cellules de code, dans l'ordre) :

| Etape | Cellule | Objet |
|---|---|---|
| 1 | `Cellule 14` | Setup environnement + assets |
| 2 | `Cellule 15` | Preparation ReDial |
| 2bis | `Cellule 15b` | Diagnostic dataset |
| 3 | `Cellule 16` | Training PRE (train_pre.py) |
| 4 | `Cellule 17` | Training REC + export predictions |
| 5 | `Cellule 18` | Pipeline generation DialoGPT (merge_rec -> train_emp -> infer_emp) |
| 5bis | `Cellule 18b` | Export metriques + fingerprint config vers `results/` |
| 5ter | `Cellule 19` | Llama 2 zero-shot + LoRA (optionnel, ~3-5h GPU) |
| 5quater | `Cellule 19b` | Scorer Qwen2.5-32B-AWQ + Cohen kappa (optionnel, ~2-3h GPU) |
| 6 | `Cellule 20` | Chargement metriques + EDA |
| 7 | `Cellule 20b` | Regeneration figures analytiques (run_*.csv uniquement) |
| 8 (final) | `Cellule 21` | Compilation finale : resume + toutes les visualisations |

La cellule one-shot reste disponible en fin de section si tu veux tout lancer d'un coup.


In [22]:
# Cellule 13 - Execution manuelle modulaire (phase par phase)
# Lance les cellules suivantes une par une pour controler/reprendre facilement la pipeline.
# Les fonctions run_* restent fail-fast (retournent False en cas d'echec).
cfg = get_config()
cfg


{'root': PosixPath('/workspace'),
 'ecr_repo_url': 'https://github.com/zxd-octopus/ECR.git',
 'ecr_dir': PosixPath('/workspace/ECR'),
 'src_emo_dir': PosixPath('/workspace/ECR/src_emo'),
 'emo_data_archive': PosixPath('/workspace/data/emo_data.zip'),
 'emo_data_dir': PosixPath('/workspace/data/emo_data'),
 'ckpt_archive': PosixPath('/workspace/data/ckpt.zip'),
 'ckpt_dir': PosixPath('/workspace/data/ckpt'),
 'results_dir': PosixPath('/workspace/results'),
 'generations_dir': PosixPath('/workspace/results/generations'),
 'figures_dir': PosixPath('/workspace/results/figures'),
 'logs_dir': PosixPath('/workspace/logs'),
 'objective_file': 'objective_metrics.csv',
 'subjective_llm_file': 'subjective_metrics_llm.csv',
 'subjective_human_file': 'subjective_metrics_human.csv',
 'ablation_file': 'ablation_metrics.csv',
 'emo_data_gdrive_id': '1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_',
 'ckpt_gdrive_id': '1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19',
 'emotion_embed_dim': 48,
 'beta': 0.1,
 'n_f': 3,
 'p_nt': 2

## 9. Compilation finale et visualisations comparatives

On recapitule les resultats (meilleurs scores, table comparative) puis on affiche tous les graphiques definis dans `src/viz/plots.py`. Les visualisations suivent les figures et tableaux de l'article :

- `plot_feedback_distribution`, `plot_feedback_weights`, `plot_emotion_label_distribution`, `plot_review_coverage` → Section 4.1 & 5.1 ;
- `plot_objective_metrics`, `plot_model_rankings`, `plot_ablation_study` → Tables 1 et 3 ;
- `plot_subjective_metrics`, `plot_subjective_radar`, `plot_llm_vs_human_correlation` → Table 2 & Section 5.6 ;
- `plot_training_loss`, `plot_hyperparam_sweep` → diagnostics d'entrainement et Appendix B.

### Etape 1 - Setup environnement
- Clone/patch du repo ECR
- Download des assets externes
- Installation des modeles de base (si `dry_run=False`)

**Resultat attendu**: pas d'erreur, dossiers `ECR/src_emo/save/*` disponibles.


In [23]:
# Cellule 14 - Setup environnement + assets
clone_ecr_repo(cfg)
patch_ecr_compat(cfg)
download_external_assets(cfg)

if not cfg["dry_run"]:
    install_base_models(cfg)
    install_pretrained_ckpts(cfg)

cfg.get("logs_dir").mkdir(parents=True, exist_ok=True)
cfg.get("generations_dir").mkdir(parents=True, exist_ok=True)


ECR repo deja present: /workspace/ECR
[patch] infer_emp cleanup nested isdir: deja applique (idempotent)
[patch] AdamW -> torch.optim (train_pre.py): deja applique (idempotent)
[patch] AdamW -> torch.optim (train_rec.py): deja applique (idempotent)
[patch] AdamW -> torch.optim (train_emp.py): deja applique (idempotent)
[patch] model_gpt2.py imports: deja applique (idempotent)
[patch] dataset_dbpedia.py SparseTensor: deja applique (idempotent)
[patch] model_prompt.py RGCNConv call: deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_pre.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_rec.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (train_emp.py): deja applique (idempotent)
[patch] Accelerator fp16 kwarg (infer_emp.py): deja applique (idempotent)
[patch] config.py seed_torch: deja applique (idempotent)
[patch] accelerator.use_fp16 (train_pre.py): deja applique (idempotent)
[patch] accelerator.use_fp16 (train_emp.py): deja applique (id

### Etape 2 - Preparation des donnees ReDial
Lance `process.py`, `process_mask.py`, `merge.py` et construit les fichiers preprocesses.

**Resultat attendu**: fichiers preprocesses presentes dans `ECR/src_emo/data/redial*`.


In [24]:
# Cellule 15 - Preparation ReDial
if prepare_redial_data(cfg) is False:
    raise RuntimeError("prepare_redial_data a echoue -> stop fail-fast")


[run] process.py :: python process.py
#item: 6281
[run] process_mask.py :: python process_mask.py
277 3733
465 4810
2920 34113
#movie: 5552
[run] merge.py :: python merge.py


### Etape 2bis - Diagnostic dataset (verification native)

Verifie la taille du pool d'items (`movie_ids.json`) + le nombre d'exemples par split.
Compare aux valeurs attendues par le papier (~6281 items, ~34k exemples train).
Persiste dans `results/run_dataset_diagnostic.csv` (append).


In [25]:
# Cellule 15b - Diagnostic dataset (apres preparation ReDial)
_diag = diagnose_dataset(cfg)


=== Diagnostic dataset ===
  run_id: 2026-04-20_04-57-10
  movie_ids_count: 5552
  entity_count: 31161
  redial_gen_train_dbpedia_emo: 9006
  redial_gen_train_processed: 56355
  redial_train_dbpedia_emo: 9006
  redial_train_processed: 56355
  redial_gen_valid_dbpedia_emo: 1000
  redial_gen_valid_processed: 6253
  redial_valid_dbpedia_emo: 1000
  redial_valid_processed: 6253
  redial_gen_test_dbpedia_emo: 1342
  redial_gen_test_processed: 7978
  redial_test_dbpedia_emo: 1342
  redial_test_processed: 7978
  rec_json_lines: 8436
-> /workspace/results/run_dataset_diagnostic.csv


### Etape 3 - Entrainement PRE
Entraine l'encodeur de prompt emotionnel (phase pre).

**Resultat attendu**: checkpoint dans `data/saved/pre/final/`.


In [26]:
# Cellule 16 - Training PRE (train_pre.py)
if run_emotional_semantic_fusion(cfg) is False:
    raise RuntimeError("run_emotional_semantic_fusion a echoue -> stop fail-fast")


[run] train_pre :: accelerate launch train_pre.py --dataset redial --num_train_epochs 10 --gradient_accumulation_steps 4 --per_device_train_batch_size 16 --per_device_eval_batch_size 64 --num_warmup_steps 1389 --max_length 200 --prompt_max_length 200 --entity_max_length 32 --learning_rate 5e-4 --seed 42 --num_workers 16 --nei_mer
epoch: 0
learning rate: 0.00019222462203023757
epoch: 1
learning rate: 0.00038444924406047515
epoch: 2
learning rate: 0.0004730447987851177
epoch: 3
learning rate: 0.0004054669703872437
epoch: 4
learning rate: 0.00033788914198936977
epoch: 5
learning rate: 0.00027031131359149584
epoch: 6
learning rate: 0.00020273348519362185
epoch: 7
learning rate: 0.00013515565679574792
epoch: 8
learning rate: 6.757782839787396e-05
epoch: 9
learning rate: 0.0


### Etape 4 - Entrainement REC + export predictions
Cette etape lance d'abord `train_rec.py` puis un passage `--test` pour generer `save/redial_rec/rec.json`.

**Important**: `merge_rec.py` depend directement de ce fichier.


In [27]:
# Cellule 17 - Training REC + export rec.json (train_rec.py + --test)
if run_recommendation_training(cfg) is False:
    raise RuntimeError("run_recommendation_training a echoue -> stop fail-fast")


[rec] mixed_precision=no enable_torch_compile=False rec_batch_scale=1.0
[run] train_rec :: accelerate launch train_rec.py --dataset redial_gen --n_prefix_rec 10 --num_train_epochs 5 --per_device_train_batch_size 16 --per_device_eval_batch_size 32 --gradient_accumulation_steps 8 --num_warmup_steps 530 --context_max_length 200 --prompt_max_length 200 --entity_max_length 32 --prompt_encoder data/saved/pre/final --learning_rate 0.0001 --seed 8 --num_workers 16 --like_score 2.0 --dislike_score 1.0 --notsay_score 0.5 --weighted_loss --nei_mer --use_sentiment
Random seed set as 8
['rec_prefix_embeds', 'rec_prefix_proj.0.weight', 'rec_prefix_proj.0.bias', 'rec_prefix_proj.2.weight', 'rec_prefix_proj.2.bias'] []

Have you seen <movie>?
<movie> is a good one
[rec] export checkpoint -> /workspace/ECR/src_emo/data/saved/rec/best (label=best)
[run] train_rec_export_rec_json :: accelerate launch train_rec.py --dataset redial_gen --n_prefix_rec 10 --test --num_train_epochs 1 --max_train_steps 1 --num

### Etape 5 - Pipeline generation
Execute `merge_rec.py`, preprocess generation, `train_emp.py`, puis `infer_emp.py`.

**Resultat attendu**: generations finales + fichiers d'evaluation.


In [28]:
# Cellule 18 - Generation pipeline (merge_rec -> train_emp -> infer_emp)
if run_response_generation_training(cfg) is False:
    raise RuntimeError("run_response_generation_training a echoue -> stop fail-fast")


[run] merge_rec :: python merge_rec.py
[run] imdb_review_entity_filter :: python imdb_review_entity_filter.py
[run] process_empthetic :: python process_empthetic.py
[run] train_emp :: accelerate launch train_emp.py --dataset redial --num_train_epochs 15 --gradient_accumulation_steps 1 --ignore_pad_token_for_loss --per_device_train_batch_size 20 --per_device_eval_batch_size 16 --num_warmup_steps 9965 --context_max_length 150 --resp_max_length 150 --learning_rate 0.0001 --num_workers 16
[run] infer_emp :: accelerate launch infer_emp.py --dataset redial_gen --split test --per_device_eval_batch_size 64 --context_max_length 150 --resp_max_length 150 --num_workers 16


### Etape 5bis - Export metriques depuis les logs

Parse les fichiers `ECR/src_emo/log/*.log` du run le plus recent et genere dans `results/`:
- `run_objective_metrics.csv` (AUC/RT/R du test train_rec)
- `run_comparison_objective.csv` (comparaison cote-a-cote avec l article)
- `run_generation_metrics.csv` (bleu/dist de infer_emp)
- `run_pre_metrics.csv` (test du train_pre, trace de la phase 1)

Utile pour avoir des chiffres concrets au lieu du fallback "papier".


In [29]:
# Cellule 18b - Export metriques + elapsed time + fingerprint config (cumulatif dans results/)
_metrics, _elapsed, _exported, _paths = export_run_metrics(cfg)
_cfg_snap, _cfg_hash = _config_fingerprint(cfg)
print(f"config_hash: {_cfg_hash}")
print("config snapshot:", _cfg_snap)
print("Logs utilises:", _paths)
print("Elapsed (s):", _elapsed)
print("Fichiers exportes (append):")
for k, v in _exported.items():
    print(f"  [{k}] {v}")

import pandas as _pd
if _exported.get("timing"):
    print("\nHistorique timing (cumule):")
    print(_pd.read_csv(_exported["timing"]).tail(5).to_string(index=False))
if _exported.get("comparison_objective"):
    print("\nComparaison objective (article vs runs):")
    print(_pd.read_csv(_exported["comparison_objective"]).to_string(index=False))
if _exported.get("generation"):
    print("\nGeneration metrics (runs):")
    print(_pd.read_csv(_exported["generation"]).tail(5).to_string(index=False))


config_hash: 214b48a7
config snapshot: {'seed': 42, 'batch_scale': 1.0, 'mixed_precision': 'no', 'num_workers': 16, 'enable_torch_compile': False, 'rec_batch_scale': 1.0, 'rec_mixed_precision': 'no', 'rec_enable_torch_compile': False, 'batch_rec': 128, 'batch_gen': 16, 'lr_dialogpt': 0.0001, 'lr_llama': 5e-05, 'dry_run': False, 'use_pretrained_ckpt': False, 'feedback_weights': {'like': 2.0, 'dislike': 1.0, 'not say': 0.5}, 'emotion_embed_dim': 48, 'beta': 0.1, 'n_f': 3, 'p_nt': 2, 'p_ne': 4}
Logs utilises: {'pre': '2026-04-20-02-06-53.log', 'rec': '2026-04-20-02-51-38.log', 'rec_export': '2026-04-20-03-21-56.log', 'emp': None, 'infer': '2026-04-20-04-18-20.log'}
Elapsed (s): {'pre': 2434.322, 'rec': 1567.255, 'rec_export': 57.295, 'emp': None, 'infer': 552.21}
Fichiers exportes (append):
  [objective] /workspace/results/run_objective_metrics.csv
  [comparison_objective] /workspace/results/run_comparison_objective.csv
  [generation] /workspace/results/run_generation_metrics.csv
  [pre] 

### Etape 5ter - Llama 2-7B-Chat + ECR[Llama 2-Chat] (optionnel, ~3-5h GPU)

Active via `cfg["run_llama_zero_shot"] = True` et/ou `cfg["run_llama_lora"] = True`.
- **Llama 2 zero-shot** : genere `results/generations/llama2_zero_shot.jsonl` (~15-30 min sur RTX 5090 en bf16+FA2).
- **ECR[Llama 2-Chat]** : LoRA fine-tune (15 epochs, ~2-3h) + generation (~15-30 min).


In [26]:
# Cellule 19 - Llama 2 zero-shot + ECR[Llama 2-Chat] LoRA (Etape 5ter)
#
# Le serveur vLLM (si `llama_backend=="vllm"` + `llama_vllm_autostart=True`)
# est demarre automatiquement dans run_llama_zero_shot, puis arrete ici AVANT
# les phases HF (LoRA train/generate) qui rechargent Llama-2-7B en bf16.
# Rationale : vLLM garde ~15 GB VRAM, LoRA HF en a besoin de ~15 GB aussi.
# Sur une seule RTX 5090 (32 GB), les deux ne cohabitent pas confortablement.
try:
    if cfg.get("run_llama_zero_shot"):
        run_llama_zero_shot(cfg)
finally:
    # Liberation VRAM avant LoRA train/generate (ou avant Qwen scorer si LoRA off).
    teardown_llama_vllm()

if cfg.get("run_llama_lora"):
    run_llama_lora_train(cfg)
    run_llama_lora_generate(cfg)


[llama-zs] deja present: /workspace/results/generations/llama2_zero_shot.jsonl
[lora] base=NousResearch/Llama-2-7b-chat-hf train=/workspace/data/emo_data/llama_train.json -> /workspace/data/lora_ecr_llama2_chat
[lora] shim installe : Accelerator.unwrap_model accepte keep_torch_compile= pour compat transformers 4.57 (accelerate < 1.3)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


Map:   0%|          | 0/2459 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,4.082300
20,2.459700
30,1.120400
40,0.243000
50,0.087900
60,0.033500
70,0.006400
80,0.001000
90,0.000400
100,0.000200


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[lora-gen] OK -> /workspace/results/generations/ecr_llama_chat.jsonl


### Etape 5quater - Scorer Qwen2.5-32B-AWQ + Cohen kappa (optionnel, ~2-3h GPU)

Active via `cfg["run_llm_scorer"] = True`. Produit `results/subjective_metrics_llm.csv`
et `results/scorer_kappa.csv` (si `run_kappa=True`).
Necessite VRAM libre (~22 GB) apres la phase Llama.


In [27]:
# Cellule 19b - Scorer Qwen + kappa (Etape 5quater)
# Normalisation: infer_emp.py ecrit dans `ECR/src_emo/save/redial_gen/<model>_test.jsonl`.
# Le scorer Qwen s'attend a lire `results/generations/ecr_dialogpt.jsonl`.
# On copie si necessaire, sans dupliquer.
def _ensure_dialogpt_generations(cfg):
    from pathlib import Path as _P
    import shutil as _sh
    gen_dir = cfg["generations_dir"]
    gen_dir.mkdir(parents=True, exist_ok=True)
    target = gen_dir / "ecr_dialogpt.jsonl"
    if target.exists() and target.stat().st_size > 0:
        return target
    src_candidates = list((cfg["src_emo_dir"] / "save" / "redial_gen").glob("*_test.jsonl"))
    if not src_candidates:
        print(f"[scorer] pas de *_test.jsonl dans {cfg['src_emo_dir']/'save'/'redial_gen'} -> pas de ECR[DialoGPT] pour Qwen")
        return None
    src_file = max(src_candidates, key=lambda p: p.stat().st_mtime)
    _sh.copyfile(src_file, target)
    print(f"[scorer] copie {src_file} -> {target}")
    return target

_ensure_dialogpt_generations(cfg)

if cfg.get("run_llm_scorer"):
    _df_scorer = run_llm_scorer(cfg)
else:
    _df_scorer = None

if cfg.get("run_kappa"):
    df_obj, df_subj_llm, df_subj_human, df_ablation = load_results_data(cfg)
    compute_scorer_kappa(cfg, df_subj_llm, df_subj_human)


[scorer] VRAM libre avant vLLM : 33.0/33.7 GB (seuil=22.0 GB)
[scorer] vllm serve Qwen/Qwen2.5-32B-Instruct-AWQ -> /workspace/logs/vllm_scorer.log
[scorer] timeout apres 900s
[scorer] serveur vLLM non pret apres 15min -> abandon
[scorer] kill_vllm_proc_tree: SIGTERM timeout 30s (pgid=15866 non vide) -> SIGKILL
[load] objective_metrics.csv absent -> fallback article
[load] subjective_metrics_llm.csv absent -> fallback article
[load] subjective_metrics_human.csv absent -> fallback article
[load] ablation_metrics.csv absent -> fallback article
=== Cohen kappa scorer local (Qwen2.5-32B-AWQ) vs humains article ===
     metric  n_models  kappa_rank
0   Emo Int         6         1.0
1  Emo Pers         6         1.0
2  Log Pers         6         0.6
3      Info         6         1.0
4      Life         6         1.0
kappa moyen = 0.920 (SUBSTANTIEL)


### Etape 6 - Chargement metriques et EDA
Recharge les metriques et reconstruit les vues analytiques notebook.


In [28]:
# Cellule 20 - Chargement metriques + EDA (Etape 6)
df_obj, df_subj_llm, df_subj_human, df_ablation = load_results_data(cfg)
df_feedback, df_emotions, df_reviews, df_weights = build_dataset_eda(cfg)
describe_eda(df_feedback, df_emotions, df_reviews)
df_obj.head()


[load] objective_metrics.csv absent -> fallback article
[load] subjective_metrics_llm.csv absent -> fallback article
[load] subjective_metrics_human.csv absent -> fallback article
[load] ablation_metrics.csv absent -> fallback article
=== Feedback distribution (Table Section 5.1) ===
  feedback  count
0     like   8110
1  dislike    490
2  not say   1400

=== Emotion labels (Section 4.1.1) ===
     emotion  share
0       like   28.0
1    curious   14.0
2      happy   12.0
3   grateful   10.0
4   negative    8.0
5    neutral   11.0
6  nostalgia    5.0
7  agreement    7.0
8   surprise    5.0

=== IMDb review coverage (Section 5.1) ===
          backbone  reviews  movies
0         DialoGPT    34953    4092
1  Llama 2-7B-Chat     2459    1553


,Model,AUC,RT@1,RT@10,RT@50,R@1,R@10,R@50
0,KBRD,0.503,0.040,0.182,0.381,0.037,0.175,0.360
1,KGSF,0.513,0.043,0.195,0.383,0.040,0.182,0.361
2,RevCore,0.514,0.054,0.230,0.410,0.046,0.209,0.390
3,UCCR,0.499,0.038,0.208,0.423,0.039,0.198,0.407
4,UniCRS,0.506,0.052,0.229,0.439,0.047,0.212,0.414


### Etape 7 - Figures analytiques (independant des tables article)

Regenere uniquement les figures qui exploitent l'historique cumule des runs
(`results/run_*.csv` + `results/runs_config_history.jsonl`). Utile pour :

- mettre a jour les graphiques apres un nouveau run sans repasser par les
  fallback article (tables 1/2/3),
- produire les figures du rapport final sans executer Llama/Qwen,
- obtenir un aper&ccedil;u des deltas entre runs (config_hash, timings, PRE, gen).

Toutes les figures sont ecrites dans `cfg["figures_dir"]` (defaut :
`results/figures/*.png`). Les CSV manquants sont silencieusement ignores.

In [29]:
# Cellule 20b - Regenerer les figures analytiques a partir des runs CSV (Etape 7).
#
# Ne depend NI des fallback article NI de la cellule de chargement (Etape 6) :
# cette cellule lit directement `results/run_*.csv` et ecrit les PNG.
# Pratique pour (re)produire le materiel visuel du rapport sans rerun.

_figures_dir = set_save_dir(cfg["figures_dir"])
print(f"[figures] cible = {_figures_dir}")

_r = cfg["results_dir"]
for _csv, _fn in [
    ("run_comparison_objective.csv", lambda df: (
        plot_runs_vs_article(df),
        plot_config_hash_trajectory(df, metric="R@10"),
        plot_config_hash_trajectory(df, metric="AUC"),
    )),
    ("run_timing.csv",                 plot_phase_timings),
    ("run_pre_metrics.csv",            plot_pre_convergence),
    ("run_generation_metrics.csv",     plot_generation_quality),
    ("run_dataset_diagnostic.csv",     plot_dataset_diagnostic),
]:
    _p = _r / _csv
    if not _p.exists():
        print(f"[skip] {_csv} absent")
        continue
    print(f"[plot] {_csv}")
    _fn(pd.read_csv(_p))

_produced = sorted(_figures_dir.glob("*.png")) if _figures_dir else []
print(f"\n[figures] {len(_produced)} PNG dans {_figures_dir}")
for _f in _produced:
    print(f"  - {_f.name}  ({_f.stat().st_size/1024:.1f} KB)")

[figures] cible = /workspace/results/figures
[plot] run_comparison_objective.csv
[plot] run_timing.csv
[plot] run_pre_metrics.csv
[plot] run_generation_metrics.csv
[plot] run_dataset_diagnostic.csv

[figures] 7 PNG dans /workspace/results/figures
  - config_hash_trajectory_auc.png  (68.3 KB)
  - config_hash_trajectory_rat10.png  (61.1 KB)
  - dataset_diagnostic.png  (62.0 KB)
  - generation_quality.png  (72.1 KB)
  - phase_timings.png  (60.1 KB)
  - pre_convergence.png  (82.4 KB)
  - runs_vs_article.png  (56.6 KB)


## 10. Limitations et ecarts methodologiques

Certaines lignes des tables de l'article n'ont pas pu etre reproduites dans ce notebook faute d'API payantes et de personnel d'annotation. Elles sont conservees en **valeur article** (fallback) dans les visualisations pour permettre la comparaison. Les substitutions documentees sont :

| Article | Ce notebook | Justification |
|---|---|---|
| `GPT-3.5-turbo-instruct` / `GPT-3.5-turbo` (Tables 1, 2) | Non execute -> fallback article | Acces OpenAI payant, hors perimetre projet de session |
| Scorer `GPT-4-turbo` (Section 5.6 + Appendix E) | `Qwen/Qwen2.5-32B-Instruct-AWQ` servi via vLLM (local, 4-bit) | Scorer open-source ; proxy valide via Cohen kappa au niveau classement (cf. `results/scorer_kappa.csv`) |
| Baselines `KBRD` / `KGSF` / `RevCore` / `UCCR` (Table 1) | Non reentrainees -> fallback article | Codebases PyTorch 1.x / Python 3.6 ; port non realisable dans le temps imparti |
| 3 annotateurs humains sur 200 exemples (Table 2) | Non realise | Hors perimetre projet de session |
| Cohen kappa inter-annotateurs humain-humain (Section 5.6) | Remplace par Cohen kappa scorer-vs-humains article | Proxy indirect de la qualite du scorer local |
| Appendix B sweep `like_weight` | Non reproduit (`simulate_hyperparam_sweep` retire) | ~10h GPU de plus ; CSV `results/sweep_like_weight.csv` a produire manuellement |

**Reproduit reellement :**

- Table 1 lignes `UniCRS` et `ECR` (2/6 mesurees) ;
- Table 3 complete (5/5 variantes d'ablation mesurees) ;
- Table 2 lignes `ECR[DialoGPT]`, `Llama 2-7B-Chat`, `ECR[Llama 2-Chat]` (3/6 mesurees) ;
- Scoring local Qwen2.5-32B-AWQ sur 200 exemples x 5 rubriques (Appendix E) ;
- Cohen kappa classement scorer vs humains article.

Les plots `plot_*` qui melent lignes mesurees et lignes fallback sont clairement etiquetes dans leur legende (`Model` tel quel, y compris pour les lignes non reproduites). Les artefacts mesures sont dans `results/` (CSV + `generations/` + `logs/`), les valeurs fallback sont dans le code (`_fallback_*`) pour garder le notebook executable sans reseau.

### Etape 8 - Compilation finale

A executer apres les etapes 1 -> 7 pour produire les graphiques et tableaux finaux.
Cette cellule regenere TOUS les plots (EDA article + runs reels + analyses
multi-runs) en un seul passage et logue la liste des PNG ecrits dans
`cfg["figures_dir"]`.


In [30]:
# Cellule 21 - Compilation finale : resume + visualisations (Etape 8)
#
# Toutes les figures sont sauvegardees dans `cfg["figures_dir"]` (defaut :
# `results/figures/*.png`, 150 dpi). Chaque fonction affiche aussi inline dans
# le notebook. Pour revenir au mode "affichage seul", appeler
# `set_save_dir(None)` avant la cellule suivante.
figures_dir = set_save_dir(cfg["figures_dir"])
print(f"[figures] sauvegarde vers {figures_dir}")

summary_df = evaluate_results(df_obj, df_subj_llm, df_subj_human)
comparison_df = build_comparison_table(df_obj, df_subj_llm, df_subj_human)

print("=== Summary ===")
print(summary_df)
print("\n=== Comparison table ===")
print(comparison_df.head())

# ---------- Dataset / EDA (Sections 4.1 et 5.1) ----------
plot_feedback_distribution(df_feedback)
plot_feedback_weights(cfg["feedback_weights"])
plot_emotion_label_distribution(df_emotions)
plot_review_coverage(df_reviews)

# ---------- Recommandation (Table 1 + ablation Table 3) ----------
plot_model_rankings(df_obj, metric="AUC")
plot_objective_metrics(df_obj)
plot_ablation_study(df_ablation)

# ---------- Generation de reponses (Table 2 + LLM vs Human) ----------
plot_subjective_metrics(df_subj_llm, "Subjective metrics - LLM scorer (GPT-4-turbo)")
plot_subjective_metrics(df_subj_human, "Subjective metrics - Human annotators")
plot_subjective_radar(df_subj_llm, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (LLM scorer)")
plot_subjective_radar(df_subj_human, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (human)")
plot_llm_vs_human_correlation(df_subj_llm, df_subj_human)

# ---------- Analyses multi-runs (exploitent results/run_*.csv) ----------
# Chaque graph ne s'affiche que si le CSV correspondant est present ; sinon le
# message "[plot_*] aucune colonne ..." ou "absent" est emis sans exception.
_runs_csvs = {
    "run_comparison_objective.csv": "runs_vs_article + config_hash_trajectory",
    "run_timing.csv":                "phase_timings",
    "run_pre_metrics.csv":           "pre_convergence",
    "run_generation_metrics.csv":    "generation_quality",
    "run_dataset_diagnostic.csv":    "dataset_diagnostic",
}
for _csv, _what in _runs_csvs.items():
    _p = cfg["results_dir"] / _csv
    print(f"  [{_csv}] {'OK' if _p.exists() else 'ABSENT'} -> {_what}")

_comparison_path = cfg["results_dir"] / "run_comparison_objective.csv"
if _comparison_path.exists():
    _df_runs_cmp = pd.read_csv(_comparison_path)
    plot_runs_vs_article(_df_runs_cmp)
    plot_config_hash_trajectory(_df_runs_cmp, metric="R@10")
    plot_config_hash_trajectory(_df_runs_cmp, metric="AUC")

_timing_path = cfg["results_dir"] / "run_timing.csv"
if _timing_path.exists():
    plot_phase_timings(pd.read_csv(_timing_path))

_pre_path = cfg["results_dir"] / "run_pre_metrics.csv"
if _pre_path.exists():
    plot_pre_convergence(pd.read_csv(_pre_path))

_gen_path = cfg["results_dir"] / "run_generation_metrics.csv"
if _gen_path.exists():
    plot_generation_quality(pd.read_csv(_gen_path))

_diag_path = cfg["results_dir"] / "run_dataset_diagnostic.csv"
if _diag_path.exists():
    plot_dataset_diagnostic(pd.read_csv(_diag_path))

# ---------- Diagnostics d'entrainement (si l'utilisateur a un CSV de loss) ----------
history_path = cfg["results_dir"] / "training_history.csv"
if history_path.exists():
    df_history = pd.read_csv(history_path)
    plot_training_loss(df_history)
else:
    print(f"[info] {history_path.name} absent -> `plot_training_loss` ignore (disponible apres un vrai entrainement).")

# ---------- Sensibilite au poids `like` (Appendix B, illustratif) ----------
df_sweep = simulate_hyperparam_sweep(df_obj)
plot_hyperparam_sweep(df_sweep, param="like_weight", metric="AUC")

# Liste finale des figures produites
_produced = sorted(figures_dir.glob("*.png")) if figures_dir else []
print(f"\n[figures] {len(_produced)} figure(s) ecrite(s) dans {figures_dir}")
for _f in _produced:
    print(f"  - {_f.name}  ({_f.stat().st_size/1024:.1f} KB)")

[figures] sauvegarde vers /workspace/results/figures
=== Summary ===
            metric              model  value
0         best_auc                ECR  0.541
1        best_rt10                ECR  0.238
2    best_life_llm  ECR[Llama 2-Chat]  8.063
3  best_life_human  ECR[Llama 2-Chat]  8.468

=== Comparison table ===
     Model    AUC   RT@1  RT@10  RT@50    R@1   R@10   R@50  Emo Int_llm  \
0     KBRD  0.503  0.040  0.182  0.381  0.037  0.175  0.360          NaN   
1     KGSF  0.513  0.043  0.195  0.383  0.040  0.182  0.361          NaN   
2  RevCore  0.514  0.054  0.230  0.410  0.046  0.209  0.390          NaN   
3     UCCR  0.499  0.038  0.208  0.423  0.039  0.198  0.407          NaN   
4   UniCRS  0.506  0.052  0.229  0.439  0.047  0.212  0.414          0.4   

   Emo Pers_llm  Log Pers_llm  Info_llm  Life_llm  Emo Int_human  \
0           NaN           NaN       NaN       NaN            NaN   
1           NaN           NaN       NaN       NaN            NaN   
2           NaN    

/workspace/src/viz/plots.py:662: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(10, 4.5))


[info] training_history.csv absent -> `plot_training_loss` ignore (disponible apres un vrai entrainement).

[figures] 20 figure(s) ecrite(s) dans /workspace/results/figures
  - ablation_study.png  (78.9 KB)
  - config_hash_trajectory_auc.png  (68.3 KB)
  - config_hash_trajectory_rat10.png  (61.1 KB)
  - dataset_diagnostic.png  (62.0 KB)
  - eda_emotion_distribution.png  (71.2 KB)
  - eda_feedback_distribution.png  (38.3 KB)
  - eda_feedback_weights.png  (40.7 KB)
  - eda_review_coverage.png  (60.1 KB)
  - generation_quality.png  (72.1 KB)
  - hyperparam_sweep_like_weight_auc.png  (62.4 KB)
  - llm_vs_human_correlation.png  (113.7 KB)
  - model_ranking_auc.png  (32.7 KB)
  - objective_metrics.png  (47.6 KB)
  - phase_timings.png  (60.1 KB)
  - pre_convergence.png  (82.4 KB)
  - runs_vs_article.png  (56.6 KB)
  - subjective_human.png  (53.2 KB)
  - subjective_llm.png  (55.2 KB)
  - subjective_radar_ecrllama_2-chat_human.png  (150.6 KB)
  - subjective_radar_ecrllama_2-chat_llm.png  (152.3